# CVEfixes · seed 43 · Notebook 2/7 · 7 job, 10 epoch · khoảng 7.8 giờ

**Chạy một lần, ra một file zip:** `cvf_s43_nb2.zip`. Gửi file này lại là xong notebook. Chỉ chạy sau khi bản bổ sung lần 5 đã đăng trên OSF.

Có 14 notebook (7 cho seed 29, 7 cho seed 43), **độc lập với nhau**, chạy theo thứ tự nào cũng được; mỗi tài khoản chạy một notebook một lúc. Loại job trong notebook này: lolo, mono.

**Thí nghiệm:** lặp lại đúng thí nghiệm CVEfixes đã chạy với seed 17 (cùng dữ liệu, cùng mã `run_cvefixes.py`, cùng 55 job, 10 epoch), chỉ đổi seed. Mono / Multi / LOLO trên tập chính của CVEfixes (19.907 hàm); các job `xmulti` / `xmono` huấn luyện trên MVD v2 rồi kiểm tra trên các hàm của CVEfixes có CVE không xuất hiện trong MVD.

**Chuẩn bị:**
1. Add Input → **hai** dataset: `mvd-grouped-cv-inputs` (dataset cũ, chứa mã và dữ liệu MVD) và `cvefixes-v2-inputs` (5 file CVEfixes).
2. Settings: Accelerator = **GPU T4 x2**, **Internet = On**.
3. **Save Version → Save & Run All (Commit)**, rồi tắt phiên chỉnh sửa. Kết quả ở tab Output: `cvf_s43_nb2.zip`.

## 1. Môi trường

In [ ]:
import time
NB_START = time.time()
!nvidia-smi
!pip install -q transformers scikit-learn pandas numpy pyarrow
import torch, transformers, sklearn, pandas
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Chưa bật GPU: Settings → Accelerator → GPU T4'

## 2. Mã nguồn và dữ liệu MVD

In [ ]:
import glob, os, shutil, zipfile
from pathlib import Path
REPO = Path('/kaggle/working/repo')
if not (REPO / 'src/preprocessing/dataset_loader.py').exists():
    hits = glob.glob('/kaggle/input/**/src/preprocessing/dataset_loader.py', recursive=True)
    if hits:
        shutil.copytree(Path(hits[0]).parents[2], REPO)
    else:
        zips = glob.glob('/kaggle/input/**/multilingual-vulnerability-detection*.zip', recursive=True)
        assert zips, 'Không thấy repo trong /kaggle/input'
        tmp = Path('/kaggle/working/_repo_tmp'); tmp.mkdir(exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z: z.extractall(tmp)
        inner = [p for p in tmp.iterdir() if p.is_dir()]
        shutil.move(str(inner[0] if len(inner) == 1 else tmp), str(REPO))
raw_dir = REPO / 'data/raw'; raw_dir.mkdir(parents=True, exist_ok=True)
NEED = ['train_w_line.csv', 'val_w_line.csv', 'test_w_line.csv']
if not all((raw_dir / n).exists() for n in NEED):
    # Kaggle có thể đã tự giải nén data_raw_for_colab.zip thành thư mục: tìm 3 file CSV ở bất kỳ đâu
    found = {}
    for n in NEED:
        cands = [Path(p) for p in glob.glob(f'/kaggle/input/**/{n}', recursive=True)]
        cands = [p for p in cands if p.stat().st_size > 1_000_000]
        if cands: found[n] = cands[0]
    if len(found) == len(NEED):
        for n, src in found.items(): shutil.copy2(src, raw_dir / n)
    else:  # trường hợp còn nguyên file zip
        zp = list(REPO.rglob('data_raw_for_colab.zip')) + [Path(p) for p in glob.glob('/kaggle/input/**/data_raw_for_colab.zip', recursive=True)]
        assert zp, 'Không thấy 3 file *_w_line.csv cũng không thấy data_raw_for_colab.zip trong /kaggle/input'
        with zipfile.ZipFile(zp[0]) as z: z.extractall(raw_dir)
import pandas as pd
rows = sum(len(pd.read_csv(raw_dir / n, usecols=[0])) for n in NEED)
print('Dữ liệu thô:', {n: (raw_dir / n).stat().st_size // 1_000_000 for n in NEED}, 'MB | tổng dòng:', rows)
assert rows == 20165, f'Số dòng {rows} != 20165: sai file dữ liệu thô'
FOLDS_V1 = glob.glob('/kaggle/input/**/groups_and_folds_v1.csv', recursive=True)[0]
print('File fold MVD v1:', FOLDS_V1)
os.chdir(REPO)

## 3. File fold MVD v2 (dùng cho job huấn luyện trên MVD) và kiểm tra

In [ ]:
import hashlib, pandas as pd
EXCLUDED_PREFIX16 = """
0017ff7238f6fa41 00221c2fbf6c09da 0025269e338a7038 00390e9db228068b 00418814617516cf 0042b92895360319 0045852842c560f0 00535c1fc068d75a
0083f2e7dc93052a 00939e03e4e70cfa 00d2e9864b2c0b3b 00e57570cd30f31b 01037de2eaea3e3e 0103c24f6149cf66 011e1a0d6e483e3e 01465d9f85cde95b
014f0b8e1f552b5e 014f442328b82684 0163a61c09658ff0 016b4eacacffb7f2 01a75da01f3e1c5e 01bb439173733d0a 01d3d0840f084d4e 022b7277d18a1c15
0239aab7fe483b4d 0261cd8408519ee8 0268e2e7ba6d8f4a 02747724c97449c4 028b042249404ac5 02916675ed3795ce 0294cf05479f2b13 02a587ad2f8d46a9
02b13625a0e8fd5e 02b71c6deac065ce 02be436194713cd6 02e5378f1ac35ecb 02e9604db34e3263 02f742aa7bf13ab3 0304436844ebb081 030afdc7c31d757f
030d3fbd01f639d6 031512286aeb93a3 03171ebaa1c0f7b7 0323d6e50e711434 03402102211924ed 034d7e0036531df2 03587f2395365fcf 035a4c52b5a79a98
035b9d08f7dcd1b1 035c097e949625ff 036d5ba6a2727145 039a8582fa0962f6 03aa913e8ad46a61 03bedb22c7609cd2 03d0bbbac1dd88a7 03d23c90bcc7828d
03f0ef1614c74498 03f8adf5cc4adaff 04059c32f419671a 0417a6a606b3d7e1 044ea5895f2ec9f0 0466bb2bdbcc6f3c 04834b092bfeb169 04888a0e5493323e
048bfcfd6e570961 04c7ad1fefdbd925 04cb3c7941862d01 04e891be1ee3a0a3 04fed2e0c1157ceb 0503063bf63f1247 051e4fc4fe6d6b94 052b5eff7395ae06
054832c32499968c 0549cdeade19a667 057cf41997814256 05a7affcc6bd6bb2 05d774c5a97db22e 05d9ab745f86621c 05dbaf7230c1b332 05def928e0ca08ba
05e0799ff8309fe8 05e8c61a3f4a680e 05efc790b9c7f0fc 05f3d2995f6577f9 0607d01aafc924a4 060e985133ca10e4 06130bbd476341bc 0613497c8789f065
06380da50b24f7e2 06495624bbf85179 0650418cd967e9b9 0674e29dc754535d 067c806d38c463b4 0687522a9b2b5883 06a356a0a8d3cddd 06a3a2c43a8b424e
06a9c07d5e34253a 06abd11e51167352 06b39fe8a6a4f1c8 06b6db9187888d24 06b77f0652456fb6 06f5581eec656a97 06f892ba71edad01 06f8cea389367ff4
0700b9df98fc20c9 0709b834011a612f 070cc8cf69a361a3 071de1b58fd496a1 0741537d5e18ca90 07437b9d626cbe2c 074f25c8d6eee337 076e8e5b8a9d6a7d
07808b9f3c81662b 078c77e4d9238778 079bbf10bbba620c 07b67e763c5c931c 07b7eee5e4c3fbd2 07caf207aa8cdbf7 07d17e9242875819 07d2c55631e584a9
07eaf8207f9559d8 07f994b1207176d0 080921ef69b0e606 081940fa32b46b30 083cb43fccde9b03 0841a1aa1c96da22 08441701a4bb40d2 0846771adcafc8e2
08687c926a5fa7d1 08935d97caf5ca8c 08a76b35acbd9c54 08c0c3a8ec89b21b 08cbed74b8afc0d2 08def32ef2a6e14f 08e2f32460b1f838 0902f4f1391adba7
090311bd7faac594 09194a274dde0804 091c66a64dabddf2 093246f2bef3404d 0941338021257961 094c2ff0c2b08606 09943854ebc8da29 09a2551f6491d1b2
09ae55d0f6d9a192 09ca84b98fadab3c 09cc16e401129383 09cf8ca7a87e55e4 09dceb5c615181db 09de7c62eada2217 09e7808ad36ce0c8 09e997dc02aaf88e
09eb4fd6f355bf10 0a15455c431f43cc 0a638d5121ded114 0a69d91314dac63a 0a8f6e3fd9493f9c 0a9f586b379e85d5 0abd0fa0113dc647 0ad891003b285234
0af49ff366d82459 0afa2bbc6a60a1f6 0b19a3be62696fc8 0b44e64cdb1e2ccc 0b6442b5df14580f 0b68311bfe22d2a8 0b6ca3e89e1c5dad 0b6dd93d1a1fc5f3
0b886799ffe8c20b 0b9e8ebc75913006 0b9e8ff817387c48 0bad2b35d9da8e24 0bb65546953a4048 0bb67ae4b3bd01ef 0bcba88c05c969e5 0bcebd467abb2160
0bd2ab903af092bd 0bfa7536c652862e 0c19d144ad330335 0c4db32885b5d360 0c9c07a68c35a999 0cc7683eb087e745 0ccc86da0fd4ab53 0ccd0ea5ee3b6a3f
0cce47cc26f96f76 0ccf99d06089abf2 0cda544cf63c7cee 0cf0b49017f19ba2 0cf8f387ba66264c 0d03cb5a0ccae911 0d0b68050be51b9a 0d0d976230f19adb
0d18535b7a41c7f3 0d3979f65ca80a8f 0d44ae7a499234d6 0d4fe93623386597 0d5b33d27ef80912 0d5ed799ce2fa34e 0d623163d83ea775 0d63f1f30add8238
0d79eb11fb60c623 0da53a52b96a6f21 0db1df1f6f809d18 0dce24d0e76bfdc6 0dd1e8394e6ee4f0 0de83fbe9f7d193c 0df2a82e16f259bf 0e0353c92f026b0d
0e396950801b2e49 0e5418755a0c809b 0e62181ed0087991 0e66b9af68f4729b 0e8a16971a65fbac 0e8ad931c29fe21a 0e910dcd65428697 0eae9b077335b6fb
0eaf221810967527 0eb7fd5ce85da162 0ef378fe31c98a9d 0f0aab61c5c42aaa 0f107a3a23299b17 0f287b1a8d653532 0f3eb36dd279415c 0f4b741ab08b1823
0f5cf968d1fa6b0c 0f608b49622e648b 0f64d5aa3071ba86 0f66b76fc6c7bfee 0f6a51557087d375 0f7086ae2d84d2be 0f7ec933a2959c07 0f8dedee3c2c0a50
0f8efd10d6a6419c 0fb6d73cbb96e329 0fb9bdc2562c71a9 0fbbca1a1bd40dbf 0fbec1796f1f37ad 0fe4a71f8150accd 0ff9fd0ba5b94af2 100165289f1e677c
10205e586a4a2a18 1034243a90bbfb5a 1054b27987d771a8 105e98a874526945 107f1c84af784ce9 1089ed7a8370f660 10978ea5959019a9 10af83d632d05de9
10b34ce0cd3f8f8a 10b3c46cff320021 10c216d5e86b19b5 10f195d1263d0538 10f4a7c192da1944 10faecc42c43f88d 110af9ec1961fd06 110b1f9c179e8400
11142b5a132fe85e 111b995999b75c07 11231ff3d0454a67 1129727425c29b95 1143342046feb57a 114bdf4a72c018e1 115d46295b8d4f68 1165b9fe1f996c46
1168841355a07976 1175f76d81f193b3 11796a53916a0ab2 117f99f6e5960024 118a655760ff2eca 118b15b01b04d97b 119bcd3f5e715d76 11a4c91321eb06f0
11bde16387abd986 11c8180f5c2dcd02 11dd311c231979f8 11e741dd21d7f31f 11e780842ce68404 11e8d8aede2746e2 11f9283135fb6c0c 1205e7f7777d1b3d
120b360e2477630d 120e9b5bc5ee06fe 120ff1e3859ca617 1225ff0f4b949ea6 122e6a5f468a168b 124b042c2612c550 12596f95a9f93bc0 12687e1db24a6bb4
1275f51357509f77 127836b6e207c628 127fa1ee6279b352 1291366a8c6d78ce 129a03d2f5cb9388 12ae357f055e6a2a 12b652f928ccce15 12bd244364fc84bc
12ca6a1d7e1ceb48 12d61d7e736bcdcf 130c182c636a9b02 131d2b9ab241198f 13370713bef290b7 134853690f2344e6 13606c71fae07ea7 137847e7f987cca3
138ed991971bdcac 13a917b18aabf9f1 13b12e518e54add7 13b785e748fd908e 13bb98f3af562249 13dddf5cb32aaffb 13e9bf46212a8fbf 14056bfdd00496d3
1405e40e2cd1acb5 1414daf48ef63fe4 14356f256e44f103 1468e4d4179b1440 147667ab79d00636 147ead7e56961451 14953927f804d630 14da2d9921483923
14e2b141a5cfbda1 14fd976c6fc633f0 150967c482ec9efd 1511b9aa74ed7588 1513c9b701e16d5d 15176fe798c2de9e 1533bcc2770a912f 153468a6f5dc0abe
1536bddc22bac99b 15488a7f5a0bc6cc 1557b7fa3a0d2170 15a0c9897a410c7c 15a3cacf24542c5b 15a432d1518dac90 15b02a677e07fcd7 15d29de38df93f33
15e29c998aa46b5d 15e69f365196d077 16511b68d55af03e 167f3541888c04f0 1683b6575f050ba1 16945904154a3ddc 16d038f06c6468e6 16e189d2e418804f
16f12b4fb3a42ce6 16f442f1664e0e6b 16fc68da13c00697 1711beb4eded3306 1715a60e6227f065 17169683b66f6cb6 1717979bed166db9 17196cb55789ee2c
1721e041ed1929df 173402fae128217f 1744cd8e9ae39d4a 174b4c84531f7e30 1760a55021d643ad 1764957d25223bce 176b63cf8a260d9e 177105d0a5e58e98
178641890d1e05d4 1789c983eb19d3da 17a4205415b553f0 17a6253c80ec2f83 17a7f366f9a0b113 17be7160c95f5815 17be75da76330995 17d3d04e60115bc0
17e1805f73866860 17fbe1d17630a9fa 17fd6456488efb44 182d080555b9dda8 185c127997fd06e5 1869676dd7d91fcd 187382d6c3a02ece 187bc7bd2e921a71
1887859d5c98c7f1 189d469d658c4a15 18ab27a8d6970afc 18adf6da9f1c9c76 18aeb2120e173840 18d48b993046929c 18df953836cdc252 18fefc1c1e60d025
19392d9d9e57e370 193d90a0068fcdc2 193ec560e8773c5d 194d068f2dce574c 194e5cb3a193da5a 19518152c3bbadbc 19640928b12defd0 19805636130c20bc
19b0bb165b637b1d 19cef1a1fdabc6d5 19d22e826ec053aa 19e453bdcb62b2d5 1a10466540cb6547 1a125dd503e2072d 1a1c1388ca41ff24 1a21f80cd8ff71e1
1a2c51d70a917219 1a39f1d3332c6c09 1a47ef7555d78a8d 1a549e98260bb5a0 1a5e7f3ab7e204d6 1a61748a5837e31f 1a644066a6244f96 1a7dd72fe0a101cc
1a7f810b3081bedc 1a8d3b1a692d169c 1a9417f46d328128 1ab64547f302c559 1ab9f18f82b578b9 1ad85ae4390b0726 1adec33e15ac88e5 1ae847ac83211caa
1b1b9f05a6b34660 1b2de532116e1181 1b4801a9e4cda554 1b7ab9d3e0f54421 1bb25c2b2e063304 1bbb2ed88fcc085a 1bdafd772575c695 1bf546e95aea96e8
1bfbc6bfc49611da 1c0472b4c7842670 1c219eee7b25ee35 1c48e6ecea41ef31 1c494d530eefe6ac 1c61e141f83b4949 1c8b8becf507305c 1cb1633a86e43fc9
1cb662bb96d258b9 1cbda186cbf241ca 1cc06b2ae4b3a84e 1cc7ed3594c96719 1ccba121b6013c96 1ce2a954fb3687c7 1ce3113cb6f5efb6 1cf97a0aeaa7aa36
1cfc536679a4c832 1d33b59a2158bd71 1d3504c66ed1e8ff 1d3e0432d9781e2e 1d469b4e98fd0233 1d4b28ab4fb2cf2c 1d52a746272d1098 1d5c6ed222b1bc27
1d5d2dba034aa3cb 1d6b586bda898638 1d7c8ff01869768d 1d86056148054ae3 1d9419d176d9c7f7 1dbfa8a02e9d84ae 1dce129f066d9985 1dfb941a94e1f484
1e026d9daa214079 1e1570584b6bec90 1e2d5e05f5408bc2 1e2d9c6b48595c37 1e41bd7df3ddd8f7 1e6644d4d61cdfbd 1e83ba7db29bbfac 1e93e48c657af588
1eafa4eab35cdea7 1ed4e534d4ebc26e 1ed5890ef9be5914 1ef5249c368d8249 1ef581a60872fd93 1effbfe29aaa548a 1f0618f30e31848c 1f0bf6c47ecb6a4c
1f2f2a1ccfa9d877 1f4a8bdbd9e84c4b 1f4d375df77f9b80 1f57885f43dc0906 1f62152c123451be 1f84c89c698220d1 1fa749e843e12d2d 1fac55f9dbeca5fd
1facf287b6ee732c 1faef186ef6b521a 1fda64c5b6f05ca9 1fe09399e68831c7 1ffbe91b347f9c08 201c659267ff3066 20254dc34df7e0c0 203363cb759df256
203e02195c940a3c 20589f83a9b11661 20613bc1a66c57f6 2062cb30d013fa10 20644f405c1b3ad3 206dd9dc0c3f3dc6 2076c0eeacbe66f8 207d096072fd65f4
20a7af39f867f1ca 20aa0df52800f105 20ccbf2465d72881 2107a81b19b30117 211b2e53a4c08aad 211bd3c499c61f72 211ff3d9489b0b35 2120d761f1ddba4a
212e8cfe205fe56d 213887e0f223f269 213ef2e320930ba6 21709a80802a6ab9 217a907af99f262d 217c752c303051ac 217f1b89f2eedeca 21828e6e6b5133d5
21a06ce37b754a6e 21a894eac53d5f3e 21aeab4e43693405 21c21ff50806707f 21c31693422bc548 21c527a4eed971f7 21cc176796049711 21d5fde2ae22c2e6
21d75b23524e424b 21dccadb39b8b47b 21f32351f65283c8 21f92b19db1b1fba 22106da8e5d5ad93 2228019ae5d1dae0 2229c3c2d477ecd9 2235eaa9685055d7
2235fb121affae90 224797a2e50cd187 224b9546a365b07d 225f4604c79c6e32 226af94e2d688578 22712304e34a8a67 22da0b80320123f5 22ec604e7b9cd61e
22f5655d6e13ac0a 22fa3a9d9122800e 231665c8a272a11b 2320314d6db41233 23276f118ea0e161 23451e0d03dd1acc 2369cb8ef8602371 236a39884ed00447
2370bba85254a266 238543423232342b 239a2dd9a2543259 23a33500486236f2 23bef0008b6209db 240c3784a1369e56 2430bc4ee008a43d 243263492473cff3
2436879e99936fbe 24369012dd811720 24412216aa449e5d 2443955f01dbe429 244fc0c317e5303b 2471db87763d130a 247924ab6bd6462a 248fd7ce4dd00372
24a17188e3e6cc8e 24b5c38d4a61189b 24f9cbac721560ab 24fd6c44b53b09e0 2506082598ebf036 25274f763c609dab 2531f2f7191e96e7 2531fb6593cac7f0
2564814c980ff4ff 256c6175f508cec5 258da4c38e53bed2 259fbc968254b6d8 25a946f1ea0ceca2 25aee79d796a9a7b 25b55b1806feffa7 25c6ca6cc474e665
25d82fdd2624ec6c 25dfe7ca87f6252d 25edb2b5d71bd177 25f127535207ca4d 2609b3fec6d36cbf 261c8233788b5de0 265382c118afee5e 265d432521e08caa
265f08d0c8e3f688 26732966b28585a5 2677e9ca5eaded6b 268454c412e336a4 26ac784227e3d0c3 26be76da2b56493d 26c09aef18fe8414 26c09c7c5d7baa49
26f6d40a7debcbc3 27250f997a9c4e6e 272768a59481b2ab 272cbb7b281dc9b8 273ec65b503ea468 2757ce8302762653 275a191de2bdeeeb 278025e8274fb066
278de0ef537f4e6b 278e2a243f4c6b13 279729ea082d56ad 27a421d1291bd6da 27b290ff1651b005 27ba8676a4df29bc 27cad22847bab658 27ccb9299f3923bb
27d3471406a28998 27deabbecdc3ef1d 282f3375623916f8 2831e26f224a07b6 2846231218532b93 284f8bbeba299ffe 28641bcdecd941dc 287f27c49b93b562
28819f85a0522c2b 289c2b5ca8200485 28aa1e50accd6e1f 28adb0ac8693beb6 28b03fdf9c669963 28c8de209294035f 28dc1babf2b2f316 2909c6862f0a0e6e
291a22709c0012ce 2923d7d704549476 2927389bc0cd8e03 2938ed01dea486c6 29433d84fac60fcc 2965241678b9306e 2983b75663e82f44 298bab82236a60a3
298c35db5d009811 299dc55f36e60655 29a6acce68f079cb 29ab613d58263b45 29c6c473a51d5ee5 29c9cf4aa9f2fba6 29de2e7a4eafd783 29ef1cc449ff52a1
29f578f8e17bd187 29fc34f57afc9c59 2a1252a01d92f938 2a27c94ccee966d3 2a305ffe07792a4a 2a3ef49c771f2aef 2a42ddd61494d36f 2a4d7127ec832d33
2a611d125ebc2485 2a6fb1c59d61e155 2a731731ac2bb78f 2a741ffbf3ed710e 2a8ca97b1e528012 2a8f04ac1adbce46 2a9a3c127198cbd8 2a9be19d4ffc6e68
2ac4be25b85b7270 2ad02977e44b0f37 2addea2ab5c0d63e 2aec581d4b88dacf 2b1b350bf6b7de02 2b3ad496c5a135ab 2b42c40d86c34589 2b55e265c549272a
2b5d48b6c3679420 2b6139dffbe9c93f 2b77cffe9bf594e9 2b78d75dd19dc021 2b7db84daf5cd3f8 2b9d4aab499d01fd 2bd3d16e93c33f74 2bd9dd045e32c9e8
2be5369c1161b1cf 2bed9578e4bd86ba 2c104c12dd864c9d 2c18846e9c2b0709 2c5266a5fca8a0d1 2c5b6eb83e48f2c4 2c5cf22fc6006cff 2c76d0fb531692e7
2cb363765a481b0b 2cb7c70e45c2eca2 2cc5bd0d285b184e 2cc73dfd04c390ec 2cc82f5737a74c31 2cdc6e601eae1e78 2ceb09e056e699e0 2d166d0786a394a6
2d3188555e9c3c47 2d504c8d5f0cde59 2d508c840af432e9 2d5cb2ecfae116fb 2d5e4c6421fcc0dd 2d6b9c9c303d004d 2d78f13960149a62 2d8fbca6428add05
2d918df68b0b39fb 2d99e6a682881cd9 2de4ba534efbd995 2de8eef1ddfd16c5 2deb8b508e844d81 2e0fcc3c386f2137 2e1d3abe6174106d 2e382ad5ccaff602
2e52afa2fc3c4c2d 2e5e5ddb5ea14c2c 2e8db00f1d766d39 2eb6b26dd997ccd5 2ece684c8be21895 2ecef76218ccbdd8 2ed398ae0680a076 2f03e4278857dc2c
2f0ce169b80bf1f0 2f1d02192db9e5b2 2f28db87adcf5bb4 2f32a6cb05990118 2f3e93ad45533d51 2f414b923c65da85 2f4fd8ced2c5d625 2f7b93863b6c3263
2f978e5d4a99db57 2fa1def6f3d1bbb6 2fb007e312bd064d 2fccc565d753427c 2fce77399a3021c8 2fe022d3064816e0 2fe1ca35078e95af 2fe9141fc6ec6556
2feb04216d4fe32f 3000779d553df46b 300f5c3a8be2b5d5 3022940dc6a842ba 30329955de4f5d0c 303b029e4e413f4e 305be62b658cb00a 3062ba059bd4567d
307a6f5ebc253f7d 309e61e9305d9cc1 30a440300e0fffba 30bb2dcf68d56d74 30c6204c1197db33 30dc515425f9a7da 30e722567870bb55 30fc3c1921d2b078
310c68450cb456bf 311425cbaf810396 31160c14d47ecf9c 311e6b84a338fdf4 31307e24db539866 31348b7f8cd285a7 314d2d362284d0c8 3166eb36bee54746
317b94f5c2c2ce7a 317fda1fc30028c6 3180eec32feb5c55 31920a365b98fe58 3192eb7f460bc5ec 31a2a25a1daec7bb 31a8b86647930c84 31aa4acc61b8c452
31c14144cb3fbb54 31c61ba8c334904c 31ce17174164f923 31d47459c8f79088 31da63671cf557df 31e5d33f801d6ff6 31efa444f5f5348c 31f0b53408b4d120
31f4cc6a593b3fe6 322023929e610c0e 325ce6bcc193f49d 326cae0b6381ba5e 3273cdaf3c3a4c7c 327f507222b8fae1 329bc2b72d3efcb0 32a171d9688ad410
32c5b3799745aac4 33003f3a4f9e338f 330f924e3addbda4 331272830ee550c6 3313467fcbd28173 331bcae8053b845b 332a991a1731e18f 3341419d3421639b
334a39df56f2948f 3356d36a4edbb306 3362582dedd8d113 336cae82097aca2d 3377deb937f36e8e 3386606b6ec8f905 3398eba7eaf32536 33a1ccf0b494ccc4
33a90cef2e05bb7b 33d82db83a03e383 33eee22d1c76081b 33f36eeb1e0586e4 33f73ec68d86a72b 3403e0775e1293ae 3408218ca21b6433 340bf3b0b4b9861e
340f7cb0d23a8b41 3410ed731e60298c 342bd12f9b61df0a 346e8b29c765df31 3498f3a5977e0c2d 3498febb9fdb2bb9 349d5a6cea2bec95 34b396bc3dd108a3
34f69f1c0e7f507b 34fdb98a02d6d27f 34ffc350dd6ca681 351d667ea8379cec 352da8b82cf1ee45 352ed7dacab280e3 352ee4f0ea1831f9 354bb76c538dd268
35760015a7509d2c 35802b2f4220dc21 3581a4399b641781 35b099537ec4ff86 35df80800cd31bfd 35e2c9b15d93d9e0 35f5fc38871c830a 361ce7641560c8c9
3626a09f91de793b 362f0040841c6e93 362f24ec23011e18 363244dfc14e8874 363e18729c8bff28 36422382e29b1a98 366145e890709602 3679a45d5c4e481a
36814f61ba1cd7c8 3689c1e4f97fcc17 369a4a5bd5df866f 36cc843a586089e3 36d9325ea3eb5125 36dd482841f792e6 36de8ea060788a37 370abcdfd8370df2
370dbde6f11d4be0 3717167306490d76 371b21dce39f4a3e 374b40bd6ab46d5f 375061081d46157a 37712e94151a186a 37743ac616efa411 377d30196ef9f1c7
378e648fdcd25a33 379f1ea4aba878f5 37a4c426c6de773a 37a6664a3cf9e977 37ab9fa012234c71 37d0fbca2dc3b596 37e2a29bd0d9018a 37e85307e3fffd42
38015880abd06ceb 3807407e7d896071 382092cdd10bef0f 38443c6d3a606bec 3845bceaae573829 387fdca12acd574e 3893f39b41209ccc 38961616bf2d5979
38b40eb4bc97ef37 38b6cd83ddb64bd1 38b77b2c23e4b64a 38c5ed7cfce798d7 38c6b456e8bb14bc 38cbc290e4ac65ee 38df8c5807a0c88a 38dfde2057a4b806
38e7666de07f197f 38f166b5af25b3bd 38fd7dd8d95da6d1 38fdf3e237562977 38ffc4494c01fee8 3903b657fa74223d 39146110757569e6 391f06adc784c078
3928963fb7e5de00 395e12fb9b062c05 39717a3389d5f763 39823794096c9bcb 3985a2c196cf0aec 398748ac1ed6b1bc 39902d61f72b7526 399d47d106547eee
39a83488a649f7c8 39b1a73223cf0c38 39b46bccd85c988a 39f906df26d8a4e1 3a0ad6ac177611c1 3a4a8d972173826e 3a5d985cee915e34 3a679c59d25eb009
3a6d32ddb07edea9 3a770976d66b6fe3 3a776d3d753793d2 3a7f0caeddacb43f 3a841c71b24a191d 3a9d03ce442f3649 3abe82e2eb87633e 3ae193d39ee5d883
3ae77bfefbbafb00 3aec8bba91d45c3f 3b01f77df70ba5db 3b06250733b7da62 3b247ed4cffb8009 3b253e315d5be721 3b31c9566945f9dd 3b5c5812f4566d86
3b67bc83b0cae0f0 3b6e05c68942fcbe 3b6fa0a63de6802d 3b716a6f13163cbe 3b87a6fab42e1033 3b8869c0af48a70f 3b930f55ba4e6027 3ba873f65c9faedf
3bc7842698905d79 3bd5439dd09d353f 3be104f159b70cc2 3bf1961aa09599db 3bf1c1d00e858653 3c07b1147c8d0c9b 3c0c3343b44685e5 3c1a145c08efd933
3c1ee952dcc9c03b 3c6d8d9706b831d4 3c79d9ad4d3fd0ce 3c82854a0bbeeefd 3c8c4171474b26f3 3ca276879dc6d5ef 3ca9ec1ef964c6de 3cb12c6ce7983f0a
3cb88d46a27ba8ee 3ccba94e051edeaf 3cde6cfa037ef8ed 3cfe82c0845249a2 3d10352fe395e839 3d27c73b5a53df3f 3d34da8eeebd76bf 3d37821ced9c7ada
3d6882372f4fec8e 3d7c8bd4f67f6eca 3d83271765196353 3d8f7897be4dbc18 3dabd8d31293e052 3dcade7da9b1b660 3de69e4354060fbc 3dea20f857d49b69
3df05a2aad7a2062 3e0c10e091d838bb 3e12add01956ea5d 3e1e99b67863c383 3e339857e684903e 3e346b310c86bb13 3e3cb05ba6a1bf5d 3e55d0f8032ea32f
3e676049455abc15 3e75a2821ab19a53 3e783aaa563ffc49 3e89907582258d38 3e95d101db3cb31a 3ec6bbb5434d7305 3ee49c7b46bc081b 3ee6115bc679112a
3eede2dfb9ff612d 3f15340538db9748 3f228e522a566826 3f4e9d0ad9bc859c 3f7dc40bac7a91cb 3f98ff80f8363cbb 3fabe119ee748e65 3fb08813a6f80592
3fc06db855b6c89c 3fde0ddeeeea6592 401e5f8d7361e809 40430e6b58632475 404db365688f7bb5 405d5a5ee35c137c 40691c87ffc8c436 4073043301a4adb3
4078f6c899b7dcff 408564861254e045 40ed726c5f6ce957 40faf936f1ccf8b4 4103e971b1424d5d 410a37e9b5557f5c 410ee8326d999bd9 411fac2588e8d240
41266231e3e68e0c 41397578052f0403 415af5090f079817 415fb758c7f1d474 41691fbcff9561d3 416b0149530d53ae 418ff5688986e9dd 41903389c762fb20
41a01bbf64bde929 41bca6d716b3e610 41e3022c0eeb9f09 41e9b550e3e4582a 41f8f8d9949eb12a 421e4fce9de7ef58 4229efbc4d8e0a70 422de7b680f84d82
4230158e0efe30ba 42374cd390175414 423daa142908c66c 4259140e93976e77 427804e5dbd68419 4285f094e1f13eb0 4287169c302784c5 42e6a131aa68a20c
42f05ba4a9b8b432 430362199b143880 431ad2bd52d50150 4345f89d8cb1170e 43726e71afa78eb5 4372a6f46ba8e405 437b31b69c3d7c54 438215fb994f3657
43acae1da58ef44b 43cae3e7088712a1 43d0e8db818e1c8d 43e9baf7808ec735 43f3252645b2bbe0 43f4ecda42ae3c2e 43fdcd388bc79f7a 44000ca14d22a2d3
440d819fc4d5a2da 441c651d42285a79 445210cb4949c995 445808d900eb2b54 445ffe5467be42ea 447b3c7f79d5effd 44a16602b4f6f1e9 44a7e5f896608407
44b1cd36283c8054 44bb23662a80641a 44d04c1cc5119a64 44ef8839055d4282 450757af15ea544e 45122f1968c82ef6 451a740c27c49bf5 4525037644f524fb
45267aae3913d867 453ac33b712de588 4549c299e2bc1f7f 455a03e2880d21bb 456cc070c5a81b0c 45998ad9748052ce 45e322ef41f7a459 45ea0c3032654e2c
45edff4f7cd5635b 46023c209fea4524 4605e8306bea3606 462e16bc40b61a04 463b6da721d648fc 465200ff42ab7bbe 466077a138322044 46638e6d96363489
46724bdc9d6482b3 46a7dcc054425076 46b0fbee3ebb4d87 46d2892c3db2a798 46ffec8848cf09bc 470685365fda9e71 472f3e6b6aba8adb 4745acd574f21026
4756c51608393a5c 475771832090e4ef 4759dc46aa63b592 4762aa406b4d16ac 476881cf3c6ff55b 47b8c72221f47506 47c8b49c458ad041 47cccbec5b782e6e
47ef611d5913100c 481f7a8d15551106 4833a0d99aae0d32 48476b2e01c4610a 48500dd86b4e86cc 485cb7b3cff6073b 4878812a18215224 487a64904a893d20
488ddcf2b50d09ee 489b881b42ea1a81 48aea63c0a3f6bf0 48b6b26c93dcd9a0 48bcd4dd978cd84c 48c8577f86812a51 48ce1840157438f6 48e653502be89ecf
48e787d5f9d02c27 48f9f67b4865c970 48fbb381c725f615 4920d6027d45eb62 492b33b25aff153d 493d465414f460db 493da866f71e71c4 49429d567b9d46e3
49433f1a1cbe2020 494aac5ea81cc2da 494b347b404ee831 4974ebce06857f78 497699056b1652ca 4983564018e3ff51 4987c15c420857fa 4997b295270baf75
49a8d84f5c4b5375 49b984cc8cc4389a 49cf8e52f4f821ea 49d453e103d78e52 49dd5fed60d25618 49e39907e13ab58e 49f94992703658cd 49f9af48c57cff67
4a2dc22598bafc93 4a389fa70e7e7059 4a3ad36a88832007 4abf4924137a4ee4 4aff195ef6c0c30a 4b3408f0b8090f5b 4b487b5814f28a5e 4b55f9e2af84b48d
4b5a42baadfb37ff 4b7394cfc8bd4fb1 4b7a4319ede26a94 4b7d57e4fae4fdbf 4b808a6e775d9402 4bbd105c3a645075 4bbf56fca9b865ba 4c14b34476ed2719
4c51657cebe0dfe0 4c5fcb00980dd171 4c61b5837d49ce54 4c6d92c22eb256a9 4c70f8652f67b0fd 4c9099bafd780695 4caa7a2994be0077 4cabef0e59254033
4cb5c674623a741a 4cbf9ac06e1fe341 4cc2ff94214295cd 4cce33cb6b2cbb24 4ccfb63ea83f015d 4cd8b1c910bd03fd 4ce420e339ac8ae4 4cf060fc1b56229b
4cfa57e16bd30291 4cfd258f3ea77adc 4d0bd700f1439697 4d2ed72cd805e91c 4d314bf3d421ab57 4d4305389e11afd8 4d735f4aecfac7e2 4d76d0980601d9b5
4d82d65e67c00d2d 4d85fcc3f000fd58 4da26cc152650452 4db9a0a6b459b8ad 4dcf915dc1cd2729 4dd436509265d8b5 4e15d59163b719ae 4e2b92c16dd1567c
4e35a488847be5aa 4e3758f7bdfbdebd 4e5e1ee047773387 4e60fc5446c0f338 4e6876f99ced2277 4e976e0bbb823d8e 4eb566925be8a48e 4eca2c52b02ce7a7
4ed8a68841458541 4ee96e53ef644a9e 4f2aa01a9d184b28 4f5d90979f799057 4fc4d67061bbff92 4fc86ada3da369ab 4ff1b1e3531d7661 5008b3452db3811b
5010724b53bfd04f 5038379c063ddd69 50497ae39442ca49 505c3c88be308888 506557fcbbb15ffc 50b0c45e8f7a3509 50bef09f29095ab8 50d3ba68073b999f
50d5535f2ee2bf3f 50f17c801f93fea9 510879e78f82bf1f 5113f47a81fe450b 512d9ec5ef56d889 5140205952357d44 5156939e8221bae0 5160a7d1b19c5bdd
516ef0bfd3fb780e 519af94f8d7a4bef 51b5dbcc27ae36fc 51fdf048eaddfdf9 52175ad309ee83be 521fe3118fd5b1b2 5234cbb424c5ef4c 523a08bee93de466
524336232e896755 5258ff846e01c4c5 526372615be37285 526c200b06cf4e35 526f78eab050351a 52745933705dc2ac 52d56991d0b2c9bc 52d6a273bdaf8a2d
530228c4557e5c89 530bca052965383d 531a8d33b2a41cd5 532429c1506764e3 53322d06c06c45e1 533510cc66adc562 533614a8bfef4574 5343c2af06562cb4
535e9c489e9ff41e 536d31cce3d56ea8 536e33b62468beb3 536f25792d38598b 53a8bd93a20922eb 5406479df1f99ee0 541c0ac63bedff13 543277731ebf53df
543c1a9aff7f00f4 545e8c596c5cbf9f 548e704f448fa4c9 54997f45613fa4ed 54b58c9bb716f8d1 54d3499fbc63d746 54d4f2c0cfc855b5 54dcb8d8faf39f42
54fd5577f1be816b 5510861906f01caa 551fbfceca10a14f 55335473899e2328 555f8ff67bd0512b 5561505faaea7388 55642c8df3d0c702 557415ad875a3f51
557955c518e12f77 557beaa0bed7b266 558692ab8e95e69a 5586d28d4d638dbb 5599428d77eb9512 55a44d2f14a059c0 55a805d21dea48a7 55b274ee9adebbd0
55ba9fa9052109c4 55bbb07313d0bc35 55bfc61b229fbf01 5605d3947244d40c 560604c7ff3dc6b4 560eabb726c273ee 5616b007d7f5ed24 564093cc31b89cd7
564a0822a1fdb946 5673f19eddbebfe0 567e8756838ffa33 5688d2d421b579a3 568c837961f88605 56934668ccfe772d 5698bf2972daa00f 569d6e9dce4d5c5a
56b071a9dc10fddf 56ba01c0803d4b3a 56f9860745b36700 57475137030a17e4 57484cad28a905f3 575c7161696f8094 5762a19945c16916 576e012fa86946d7
57780a5b9b282fa6 5781e445240a68a2 57a2287e082fcf21 57b06f64fdcbeba0 57c28b1a9bb1e7a9 57c31bf61672598c 57cfbe9233724917 57d70cfac4cd0f94
580e760c10d5f0a0 583243f9c136c2a0 583809fc63274fe6 583f7560bec943d5 584689982a0a8a4d 585b894a70287f5b 5868f139999a85cb 586d412ecb842065
58737768ce016284 587d0ddc70b9ec17 587fda5552d56a01 588683aed694e67a 588e18d6b285bc5e 588f520da5b7bed7 58958354b94f4645 58d60ccf65f7f8cb
58d9a4b32bc8f167 58db68c4e09e7ae9 58ef5583b6600440 58f83d47de3eb46d 59243c77b3c393d4 59286f0c405d0b94 59324767cd036c51 59362e8763433e7c
5937f9bade9bc238 59410d1374c795b7 59505918e4bfb273 598bc78b751ebaab 599d7adcb93e58e1 59ab7f66c904d3e3 59afeb6b3b9866ee 59c7a45c340aaef3
59f570aa4ccbcf82 5a05c89a525753c4 5a0802eaa828d69d 5a2d27c220c1dc98 5a69ec9e64914cd1 5a8221b16b294b52 5a99710bd01cb827 5aa31111d79c60bc
5ac35accd8e4b112 5ac36988e8ac9496 5ac53cb5ea1fe5cc 5ac6c2f56de11126 5ac9a684d76a60a8 5b027eaa6bb30555 5b0aca100f88feef 5b21108648d222c2
5b4ba38518a6be69 5b578d2b52500759 5b6f876677791a14 5ba91630eab1bf8a 5baa55fe6030971d 5bb00cf72890b360 5bb5855de394bb2c 5bb76c58c41710f9
5bb9d52d4e64ada9 5bda74806d31accc 5bec49d5ac91614b 5c20a0a7d6635d9d 5c248d09de443b27 5c50380eca948c1a 5c50dfd3517532af 5c7232c42c78d040
5c74d5d91fd9c0bd 5c7a5eb224da6f01 5c8f9e6aa25e1c49 5c922a8449ad1617 5c9f9284151ed2a8 5cb4dd8bcb674a9d 5cf85644d11cca2c 5d1bcf60cd2a5514
5d20c73767c62c73 5d2a43974e3827e1 5d315e8b6e209478 5d4472275cffc62e 5d552ff024b93feb 5d5780e44cb398d9 5d68da1e2e669a90 5d6caa3d6e07b154
5d745165c5400058 5d757f16764f4fd5 5d7b8b6fec53e09f 5dc30e0af09ddb94 5dc68015f4681e14 5dda2334dd4b82a0 5dec5359863a5e8a 5e1c89162cfd7c5a
5e21b49b377f703c 5e2cfaec10b7f868 5e41cf020fe39516 5e5e2ad08bfcea1b 5eb6f3bcd39f4900 5ec71f422fdf654c 5ec919f857897582 5edae3789253aa15
5ee23387cbb178bf 5f025107db11007a 5f4a854fe87dce8e 5f653f52d442bc7f 5f6d81cc715b9a95 5f745cf3bcbe3c72 5f7940fffeffcadd 5f7a22cc93a1ea93
5f8e170f073157da 5f9698cb6a6b410d 5faf44d15486d78c 5fb3fee381d62f7b 5fbea2a53c39c7eb 5fc943baf0c6d2c1 5fd6f61e33207b36 5fe51b8f9e2bc5a0
5fe815485baeff90 5ff122ca633a2741 5ff7523ee2d50e19 6000146e755c4376 6018a559540028e6 603c3a3b804513f6 603da764a5c81679 60485632f177e3e1
605c907d05738b63 605ffa65fa9639ae 6064561185530e73 6067d0e88237f16f 607eb0de31ebb479 609405f9cd0652b9 609b397c9df07acf 60c0b5d24f79df7c
60d6bbc5c7cd1d18 60dba961bb449de5 60f7ce1fd7da5f48 610651eb30308eb2 6118f14aaebc7272 612ef6485d84570c 61401d3353c583b2 6169ce79d873b174
6175e5829c0b0558 6183931a38105738 618583e553a4894a 61a0d9d435e20a1f 61bfe9e893d019be 61f06d15eece19fb 620c6b652c61c408 622c92a558ebab34
62392cc0a60c0eec 624ea0c350d31ee6 625065957cabae79 628380f737b49915 62896f8184a4e212 629837544118e40f 629bfc29575c6def 62c58e10767593e0
62c66f6d448fdb14 62c677e70781f7a1 62c7fbba916ee8fc 62f54026fdd78ea2 62fda9d466769af2 63292695aa6819eb 634069f3edfaa642 635bd7608cfcb1d2
636d92e0523f9df2 638b888a852ec72b 63a50d1cbf274d0e 63b30043e6c323ad 63d5898cea0a6deb 63db9a9861fd71a0 6412e8118eb7b094 64134823c4772ad3
6444cab96cbf3a9e 644ded4b7c33a4a1 64506f988d1257bd 6479cfae30732c1c 64b78e59a561f802 64c7f09a00b3d181 64fcef008a1190fe 64ff1bb600cfdcbf
65134ed8b10fd7a3 651af3b284470d1d 653045a35a598972 6539bbdf3ed0ca87 653c8cf0f0a68643 654c01231751b70c 6554d59344f17cc7 65635b22bcb7accb
65a4a7a6fe64608b 65b59021224e65fd 65f6c5c9e2b62e61 65fb0ddca14cb5d9 6611853c3aa833b8 664403d52d91203e 66542cab7bd3f7b3 66893a4097f5e388
66900a1c2f3a1217 6693ce2b8db79637 669407d8c0d3c4e1 66af4b54ef7844bb 66b77d5177849a2e 66c3b0f1af6d5705 66d8cecfe34edeba 66dbabe3d5d2767d
6728fdba163ccf51 673b86fdd7931c6c 6745759e38664715 674f9a018900a68c 67615ffa06f605a4 67783e383074ec56 67a824ffe1c4b362 67dda69d56571e75
67dfaa170288cad9 67ef806f13168346 67f933b1221a46bf 67f93bd95e8a4f0a 680b43a94d6257f4 6811d05091d67149 68564ded24cb12c7 68639d0dc982fd6e
686afedb1eee5175 687194859e64794b 68954f4c33a8ec42 68adc0bb8e837163 68b416a2cd6ea27e 68b6ee51ccceecb0 68c5a810f1eaa328 68d27153c8a3d38c
68e2ef0b9ef096a0 68ea8171b6ff6fd8 691299078075a537 6931ec21e47b780a 6937bc85a81b40fc 693df2f9c1211a1a 695f294669d33c67 6969b13e0793ac71
6975fb5be8ca8e4e 69858e7ac540a846 69a3f48e0f4f5d15 69a9e5d32fd76d8d 69d8990f0d3cd87f 69de6e08107b7fc7 69e6ae21aec3c13f 69ef1c59b50feb3c
69fe9bb78b8e1361 6a2d1b85e1b549b3 6a3bdde4716249ec 6a457c6835860a1c 6a525982d94854d9 6a5bee2f5b600ab9 6a62f68222bdef6d 6a65812e2747f86c
6a95db9562f183ba 6acb6f5bcebcfbaa 6adea8ab6d23d30e 6aff125e5ab057b0 6b0dee5e39ce164d 6b146cba553083fd 6b23b40006caddbc 6b559dea6977eeb1
6b610ac344059d34 6b64ddef1c3136e2 6b75d381befa9bc4 6b80f7cb7da2140b 6b83643108ab753f 6b8f4ddf0f21fb5d 6b9739239c661ff5 6be440e983cbd152
6be98c8060d72108 6bf3bb79d7a66535 6c0710a59b5e5d90 6c0fc0770cbb4835 6c190448efaac01f 6c1923524d284365 6c1b6a7e0dcedee2 6c255c3d60b8a010
6c28e0f04d110212 6c59e146ac0ecf2a 6c6a6516d125a9d9 6c6ada77191e6481 6c705d1490e238ad 6c78bb908dbfd72b 6cacda89c17b4dd3 6cbef8a83287c615
6cc3ebe6951d6c85 6cde4b3519d861ef 6cebc3c75c64622e 6cf3be7e772f4b74 6cf78203b5d0ea09 6cf80c8fafbf178c 6d0edc6016d04195 6d1b7c74b03c26af
6d3c978f3558b50f 6d4a569f0fb2b431 6d4f9b46850b87e8 6d5b94e84d1eb9b3 6d5fd9da9ab38698 6d6d8d95288a38f0 6d70e50ce0be0a8c 6d7de43c7a8eea59
6d87708fdbaeadde 6d98d73e2f2b324b 6d9b7165995ff80e 6db29d2e06a2c64c 6db9aceffe8b3b9e 6dcaee8e51fa9f9a 6e03af1e93dab3e9 6e15900165e51f71
6e2a4f0a0e8cc9d6 6e329c6fe68fb40a 6e36c98b063608e2 6e3fa365fe7041ae 6e4009c129d2f26a 6e4dfa2347b51628 6e67007c4177538c 6e77466d383c5b67
6e83a08c0ac306f0 6e86de6377eccf9d 6e908de98fbaa67e 6e99239947ebb8eb 6ea9e765ab9c05f2 6eaf2c8bd0160daf 6eaf80ce25df3788 6eb9ff1169c5eea3
6ec2ae5c4bb6c2c6 6ef88f9e4b5d0234 6f2c420066ce345f 6f6253346e67f2fe 6f67d3e3bde12644 6f71202b1787bc37 6f7e74bc151af320 6f831d510524a847
6f89957511c385bb 6f92e6c46e0cdba9 6f9873a8c3a1d2eb 6fae3602b698035b 6fb913f7d8180b75 6fcd5cc85f5343d2 6fed7931fbb51d2f 6ff20138807a84c9
6ff9687eaa3dd143 70074d96980d3d40 7018c0265ac3d5bb 701fb69b74196580 7020d7b2451e1e6a 704473343cc8bac1 704e73e6980e548b 704e95201cc7af6f
705bfe75e727d48d 708ad3029743a7ab 70ad6cc71f60ebb2 70b2680b05de3e60 70bfee7a15d4f3ac 70d1a4472bd5eb41 70f0a6ba0abe13ce 710dc60efb9620ad
7112d6d794d91e44 7118f57f0966c9a8 717545578dd6a0f1 7190927ce8115597 71b22c17e42f99e8 71c32885bc2f6baa 71c8ecc09a4ccd0b 71c9b0d342579bf3
71d1ecb6461b4878 71d3fc246ee744d2 71ed08b13b13a46a 7213b034f692b773 722a0b4302d08b9f 723af253ec26d567 7256b9e1780224a1 725f360242ae9ba8
727cd14998c50757 7283d5540eb1224f 728d1f69be8ae3d9 72ad08941550c18f 72cdd46aa283da44 72ea9a85e19ee04e 73086d52610bc80a 733e1c01f8195909
7341f187020fa6e5 73452ae686a8c07b 73458c8d39c8946a 73479964fedc7d49 7362e34d207fcc22 736b9edb12c914e8 737241a64e85bfa3 737782032e8c06c4
73878885ca53fc32 7389d0dcec6213f6 738e957b328d133c 73c6e313672ad3ac 73c97ce41ac5aa7f 73fb5ec629c2514d 73fe93ea50c1831b 7405ee821e66276a
74143acd8d31f5f6 74315081bcb18df5 7436bb67a0057dcb 743770c031d55978 743bc37cc2836e74 744c50e99fa64d91 746212aba747537d 74891450d011b412
74a3bb6ef464b84e 74a456fd535e9216 74aa447c46a14590 74ab104334e79fe0 74ac8ae83e42f577 74b513072ba19f13 74fb0ec23046f1f0 75221d1a30d92c3d
7549b302c2133428 7553baaad726d7f4 7561cf1688880cc8 75a08001219fdf24 75a715b5cb8fe58e 75bcbef83e3797c8 75c26ff8aa62a2bf 75c61ca0127e00d0
75c6ea89e2314188 75c9c221be5ed148 75ca1a257146d0e4 75d9f99acff30407 75e1c9238aec776d 75f203e43a4a6113 75f26e5b64c02501 75f7f4cad2429480
760529e7fe0fb9ca 76351d69649060b2 7652762ec0e49ea6 765ab0d448374eeb 766d82d69f91048b 7689257ae79fa365 769e57b680846c39 76b89ce0609a205d
76d44cef3463d467 76ead0e9ddc4b5f9 76eb83b7cad8a019 772fbe0655c08726 773fffa4aa959fa1 77447f72fc72ff5d 7771b1d90b647ba2 7783b629b260a5ed
77971fe9ca68d99e 779ba50d1c03e02f 77ad1f2cf7032b0b 77caf58f0b548757 77df91c16da3547b 77f25c9c91ca621c 77f428a4a9ca74c6 7804713b5d742a73
7821bbd5d5e640c0 7839fd9deefec483 7845c514db848ef9 78564f6020e680b5 786ad8753c26ce29 786c3bd22d56ba84 787ba0f7da6d8955 78a66b4e81007313
78c8616070d59fb9 78e2f8fc3b26f3f5 78ea1454e16d942b 78fdfb15a498a345 78fef448b57d1f47 79527211d732a081 795fad7186a414e6 796d7e9d0c9130d2
797d962bda69c424 799a90ceae29e2cd 79bba8f918cf649f 79c83fecf24c1ee8 79ca9d3ab073d561 79d02ac6a12b011b 79dbfcc5e5a46a55 79ef0362737fbed8
79fb195dccf830c4 7a05460f1b154a79 7a1e684b49707606 7a2225740fa6477a 7a826a2aa0eebb85 7a9364f577dd6e4f 7aa420db6b7fe151 7ac3fa8a1ae584f8
7ac781a5da3ce370 7acf06f8bfde680e 7aded2400be28738 7ae8370f72a8c9bc 7aeca55b56714bca 7b03468581e23e4e 7b076b5f7c97d844 7b089e97ff1691b7
7b09217a80d021db 7b181043dafedccc 7b1e22fa8545818e 7b212d31f8292228 7b24cb41195fef7f 7b286746649aed69 7b6529c15278d6a4 7b71fddf83ad4a3a
7b84943128262d43 7b8e91862cdf6462 7bb6fe13966b0d02 7bbeb102c75ab889 7be603f541bb43e8 7be8ceea4c22361a 7beae8a1a99ed9a2 7c0557b4e4b3a94d
7c2ca09c848765a6 7c38e086e416fd79 7c5bd6cf2f9e0f98 7c5faf545dcb7921 7c63271a96c0cf4a 7c716d4ad8e577e8 7c765053c469614a 7c83770a295d3f78
7c857ec465f6529f 7c922c69c5a81cd1 7c9777ecb3caf62f 7c9b2b763d5cc67f 7caa6618130af084 7caeb12f6ae08260 7cb616dac6dcb3ee 7cd3a7b2bec1f63f
7cdeee97f3d73a25 7cee1026e5c9c440 7cf40fcc3c5fffac 7d00f6e20002f5b0 7d05f96a072d4f3c 7d1ace6246f743ce 7d31dd3dbe6064f7 7d63b21efa8f5979
7d72f95b9046da4f 7da170c8a9204ddd 7da9cad00b13cfd9 7dabd656bf0ca8e9 7e1e94e8c091df97 7e2249dff03963ec 7e407137c1575569 7e62111d1c4215ec
7ec4fd0d577e890d 7ecdb9e39c2096f2 7edd8945e84f702d 7eeacd7209984cd9 7ef353a3a360b653 7f1cc302adb2b46e 7f24a89ac1bb4e67 7f39d15648550c79
7f604fc8a3a5e298 7f64c77b728efa34 7f65ad252192126f 7f819b764cdd9f32 7f8f095971e34f71 7f9d0a0c10c8ea49 7f9f549673a75662 7fbc6d89f6b2087e
7fc714c5b9482df8 7fce86ffeaa3865f 80005c07420a2f24 8012502c79ab4347 80326c48bfd72fb0 803e077bfca49e88 804702094eb350d6 8048996a429b7525
80563b3e91cc3a2d 805e14a214f0ee0d 8066b9df8de81831 806beba432b58538 8076105f69db7309 807fed9c7116b073 808cfef0dc1d0693 80a2587c64b4fa46
80fd17470a3250ce 80fe88b4fbd16bd0 811da7bf7397ad66 812151db0b57356b 8127713b70d2f37c 812c7223574ca342 8138a39a425f6f44 816696ce1883ec72
8168a4c670e254b4 817a1dc66bed2f58 8186f7ef22fead00 81c421bb24815b4c 81d207be7a1b6a35 81d722839aed5fc6 81dad014e2b27177 81e8e7ee8847f98b
81ed761417cb4c4c 81f03dcf56cdb84a 81fff7e0b36e8a24 8209ad474f97084e 82181f41d9190135 82258b076ab91443 8225c7733faba450 82293075ea838688
822fb6b1c5d4875e 823792c82188cede 823dd260c32bd110 82771825ae6e58a9 8283b78f8625825a 82adeafa1fc46eee 82c70c09dabac01d 82ccf63a4992bb61
82f43d1dabc831ff 8302753e399ae8f4 83133ca5e6b63e6f 83137bf7e9486f7b 8334fd3ee7ae9b50 8341df5e801c675c 835ba0caa621f8b2 83685f4d8d2c89f1
8368c59c8db4fc49 837767e4222828bb 8378a79eb7299f7a 838df45af74de7dd 83a5a1417412a89c 83cc38829596cb17 83ccc409243fa838 83dcb6e5694aa930
83f92c8e63dc8a05 840717d794dd9c40 84187bdcc6f8be14 843248d0c11b0059 843b4b692ecc49ab 8448a307f3612ddb 847be60225e65a51 848780ffa0d4d633
84cb4fa28fd9dd28 84cc9ff3a58dd2e9 84d2a925e40589e3 84d2d5b47d8ad820 84f1f6f979419f18 84f72af3181f86c9 84faab940485f77d 8510bfd3637399e4
851c0233f673e362 85297862c48727e4 8532c705568516d1 8543ee71f5601452 854b90d792d8e81b 8557e958d2585870 8557ee30c284426f 8561dc58eebcaf71
85655ca1eaf01aaa 8572a663efe95eee 8598c61b127842e2 85a9249db2f785d9 85aee8fda6b2a4af 85bcdf382cd90065 85ef424509c26ba9 860c8b9248a225f9
864830d329692692 8672164e007b371c 8673c3c2b8254016 867d4d42f2abfe56 8692357e62ef934e 86a744e7cc7d979b 86abada73a7832f4 86cb4dac1a071b00
86e3437a6018cc34 86e7e270738421c0 871483b41a421876 87152fc7699492f9 8718a275bc3911eb 871f28d326c4dd82 873f6919a486fbca 8743870f0b770162
8749a79544aa2a4a 87913fe13fc6ab84 879306a1f51b9a66 879366bbc81d5a39 879cf94615e8ef11 879d95fe33b16cc4 87accfcb156a3e93 87b1c1b7ec6c6ac9
87b41671a1897a5d 87b7d875cf41bbde 87b8d4448b5c3176 87d2870098af0cb3 87d643fcb3dbd0c2 87dfd7b4466f7fb1 87f488f2569672b0 8816afd00cf7c717
88217be6aecb1ec4 88219f31910bb80d 8830968604d91f4a 8863b56616ab7a39 8864c692669d7736 886618beea61a996 8880b176059db129 88999b0a4c99475d
88b80f42bfd8eb94 88b8564c0b488caf 88b8a6a781b0b810 88d10e4fce155298 88f259aeae9464c4 890c477927b9a800 891cab170729097a 8925f3446b89789f
89310ef1c057bd7f 8938c069d0360314 893db7b22db8b29d 89717e41ba2356ad 89832c9ad52886ff 898817b377251781 89953abfa211a3b6 89cbbce92ddfee23
89d4ac602d552c86 89eef8a88f6d9cd2 8a0080ea42d76900 8a02625c5c8a2091 8a0a806fca5b8c1c 8a25cfe57ae95fd0 8a4d7e015a3357e4 8a4fc04633965bd3
8a6adc281c4a57eb 8a714ba98bfe2535 8a77c68d544f1566 8a7c7f5680e274bf 8aa4376fc5abef7e 8abd5c807fb45a09 8ac4c4d241250bb6 8acc3ae6d05d47fe
8ad75ced4ca63560 8adc8fff7ccea385 8ae82f76f5c005cf 8b01d6ae54255129 8b09839d08a38f5b 8b219879b790d724 8b3a316c84934c47 8b467ef145a246d0
8b681f1337e5bb16 8b6c7aae590b5f75 8b6e0afbc60fbc0d 8b7dd6cee51a0419 8b9b34f417c021a3 8b9ba15481b0783b 8badb2576f382f69 8be03f4d5b410daa
8c035dd489e6762a 8c0b01e6f5818ecd 8c1439290c2c965c 8c234b8a8444e3d5 8c2b7a5dfc0837f8 8c2ff020adc0e168 8c4d5ae8dd9a0c06 8c5ff448138b5524
8c6b1319a76f52cf 8ca64fb87a39b424 8cead4990203fb54 8d24f2760e75a15f 8d4416eb609017c8 8d44bf38eadabfbf 8d58443970eb5e85 8d63e8f30b3dc43c
8d6839a8a2b001bb 8d93529fe0e4c542 8dbc27aa03ba592d 8dcfa4454a1019e8 8ddc059284cc8ada 8dfa3f9935582f18 8e14fdf627df3a66 8e1e0cb950c6d79b
8e2219edbeaee388 8e2a9d4a75e2bc67 8e3563f0bbc5f3d0 8e35932c5fc566a5 8e44af747617339d 8e50cae952f9cadd 8e5cd682c49855ed 8e676c7c45aee190
8e6f2e5badf1613f 8e792f28561c08cd 8e80d8ad03b2ea1f 8e8a148a6a4b0b79 8e8f89773af1e7d2 8e9bbfe3f8d4e3a4 8eb82e5ea6c7dec6 8eca69d4f6929c50
8ecd27037a0a7a58 8ed15f6b4186f4e8 8ee779769d32bc99 8f0abdab71252ad2 8f1eb4aaf7a5dfab 8f31cf856e944eca 8f3befec788eac61 8f479462cbaf63ed
8f54536117f27485 8f5c74791cacb726 8f7272b05bab6a0a 8f81589a4b6c2e8e 8f8b1dd69a72129e 8fa10e09c5e8b571 8fb949b496b535ed 8fc3c658fea06d1e
8fc69cce3a925c57 8fcea05916ab87a4 8fe9a843b5f75bbd 8fec3559f41f481b 901f5c74adfcfe23 905034e7f5bd7f15 9061511969ffdb33 9061bb6be356c4d5
90abc03cf4f2b8d1 90b090d173092814 90c1aa2e37d95351 90c90e09276ce67f 90d0f0bede80cbf4 90d5b59a3fd2cdee 90f0b0080fa57a20 90f4debee1f6eae4
90facd0082e5d0ea 9131d43282cc8606 914edf4816d2c559 915a90292003c31a 916a506f77ad8f0c 91790b19f7cc5ac7 919f4c95654977bd 91cf55de92512a9d
91d8d6249e6b632e 91e44c2ddf5be8d8 91e49676c5490030 91e4cb78573aacba 91ebb986f12c8c19 920269bde4e7cbbd 922c9e45a7589617 923990ff51c0b534
92587f4c0c425b80 92716697366bedc1 927c69a91951f0dd 928491b91cc47ca1 928a39b6c2247a5a 929275252f3bbb9b 9299ccecdefa5f10 929ef33030f205c4
92b9b915275d3ab1 92bfbaddb1c32c13 92c7c6fc6c773277 92db8dea076ab94d 92e01e8b66d594a0 92e1a84b1355f06b 92e6a2aaec7ae022 92f4b3775fab0d1c
930bd6c23a969153 933128a60065d645 9343877701140483 937023d5ac9adad9 937455cf5a017906 9385606864a7fc6e 938a11b693565c65 938afe87877ddbff
93aa566989dc1e04 93c5469411fb5230 93dfbef732a50b6d 941287967e8e3c01 9427a1cb2b3ff5e6 942b2c99599511b1 94416f18c1c4fa9a 944ecb07135fcbb5
9462981a2777b34f 947e23acc9e53c0c 9487b1d1ee634fb5 9499b86df4454357 949ce79b8649d0b5 94d2908335bbe9dc 94d5cddbf78c1c75 94e30cfc4694c32e
94ecf86b67736c72 94fd088551a70976 9503e092c78118d8 951b3c4259733780 9520cfe6acaa4a4a 952dbea18b784b88 954f0367b180ec7e 95699f24b896bf57
956ad3c414963cae 9581fadaf7bbd87e 958828d74958695f 95a226860e6f5b61 95a34c7092d8b425 95a392c6a5ea903e 95a40353218277c6 95ad7a8e732a75da
95fbffbccc19227c 960813e9fb4c76a6 960b8b874321aa1c 9629887193e37f4f 962f502f62238894 96344526cdd7ec7e 9634587b95d07944 963bb9f86a0e245c
965b15768fbba19b 9662498a13bf2f08 9669602f5148f9ad 967234a39f609b89 9684d1cdb85b8525 96886f12776bc051 96b55e3fa1add54d 96b609c3743a1e26
96bc973bb5cf7798 96c59b730141d26b 96ca7d31e5f6cf46 96d6c66cb0f1c638 970eb25b4b4985f3 971c8d4c66b86401 9723fa00399b124a 9726398708c6efdf
974c49d3fe134efe 9754fcfc34db809c 976499ef856d0372 978f3462d0cc3835 97901e8763107624 979c328aefa7d391 97a1157ab16cc81a 97b908428e377b05
97da6bbe5f9bcada 980201df297d18fe 98176d586d13ac2c 981f11d71e650ee0 9825558b0b5b00b8 9834a6056136f0f6 9838f1c47758f7c2 985291a0fe3d1bca
985ce570f4296187 987ccdba9a366c82 988d4bd65d3c7945 988dbcf46ce87dc1 98b533bdead49ffb 98c7de360ccef26f 98f439b7394b85d0 9907763b19c02f03
990c3b6eac79d4c6 990c97e2707e7126 992ac95ea67f818e 992dd14b9f9083f4 99338d1236a9764f 997a5cd6c9840ab0 998e873e6c249b5d 999b189b4bb2ff55
99c1eba9b9eb5c2f 99d4408233ff10d8 99e560aac7fd6cb4 9a00cda8e91aa3d3 9a24ba2e5b94be16 9a2b4bbe766b4674 9a437f3a69eeae56 9a46af1eeeba9ee6
9a57e0510b5d4c67 9a61f23cfecd2048 9a73a6c19aefed7a 9a83b6bfb2655a6b 9a87092090a5dcfc 9acbf1ed82106763 9afda4dc1d51c388 9b0dedaa9574393a
9b1df8d27cbb6403 9b20d02c3cc9802d 9b2146f60a372f6e 9b452e34e307d913 9b4f2135ee4bb135 9b60672b67d294d8 9b60bb502f5970ae 9b640a85c447d697
9b7becb526d82536 9b873c6ad4559929 9b91cb10f3bc3339 9ba2b20ed3020f10 9bac6231a4d1c2eb 9badfee9968ab304 9bae897c2620ab1f 9bb20bfc264b81cd
9bb73033194845bc 9bca03e7b6193b26 9bcff63be39295c0 9be8bd065241c007 9bf53381c3c9da17 9c04aa51a9e8822d 9c1f0fa326d978d7 9c1fe8deda2e6311
9c237ffd75950411 9c3f85b072c8e0db 9c4c4f5ff0aacbfa 9c5c49d7324441c5 9c63c522a6c55b91 9c93709b51bb70de 9c9d532b5e66e855 9c9e3c2ed73f6d98
9cb694838edf1330 9cbdc0a7ad57bab5 9cc57a0520857223 9cc93c6579208f59 9cfba2ded002dc90 9cfe5096369e6348 9d1e4200f367a649 9d2174c9100c5083
9d3010b6809fdb9f 9d3f35d212e4676b 9d3faf8568ddc7f1 9d5869c82f42b867 9d6788a3adc24025 9d70ea75706b93a8 9d71ad7129f8b90f 9d723d6cecf45c81
9d75a8082ddae29b 9d7ad8745e07e1c3 9d861d6c47da6327 9d898ef791851485 9d9045356f973bb1 9d95ca445396c4cc 9d9df89b59235dfd 9db7cf8e3ad090f4
9dbf920eab4c9948 9dc19bceebc1fc9e 9de47ff4dfce5251 9e317914d907fb2b 9e3201e8eb7af894 9e334b72fc12a88a 9e771ed04a31b0dc 9e8ea4bcf221f49b
9e93475ce09c8afe 9eb48105de6660dc 9ed60bd0fda8fe66 9f06e07c589c2ca4 9f3c66a073a58d5d 9f3f6ec12d6062b0 9f4a99f1bf80b7cf 9f5514a4c3a1d604
9f6106a1bc856644 9f65816d8d3523ef 9f8aaa15c5985b54 9fa824466ded3c62 9facecdae26449f3 9ff67b18eca8f22c a00a0d6b8f7b68d7 a0297fcd386c5fc4
a038e879d3d096a6 a03db7830741d246 a0496f5576400053 a056d8546dc7c316 a07c5ebbe1508707 a086760c315c352f a0a33b4ec0062f19 a0a92446c2c81a5e
a0aa7d10c398fd2d a0af0aea6d496c83 a0c3aa230432ef97 a0dffe87975ef7cb a11f26eb69fc14b4 a1257e81d45ec594 a134c9ce931d9ada a140314859704fb4
a1704f66ffaf2ee1 a1908894b40f9cb5 a1a4d07bdbc9ad46 a1afe6481ec7072f a1b7348c632dc720 a1bd404e16e91f7d a1c517e9a41515d4 a1c7f8f3766341d3
a1e18807ea7083c2 a1e55e7ea5b28ecb a1ea999f3cde7107 a2053e802778f1f5 a210b560e13f4b30 a2127610d5661eef a225c0bc7805e5a6 a23aff6a28c15696
a25b7796601b11f1 a26de6feeabde6bf a26e308ab7e52ac7 a279c3736492cb10 a27b53ead0fca35e a2811d9d36915f6e a281639ec11e23bc a2a60193a18e3e12
a2bb897914d89c68 a2c2a28c49e239b6 a2c3424878a32cf2 a2c4d404f7c8672d a2d35533b27f8706 a2ddb4ffab016106 a2e2ceea0e2ed40d a302a7ea3245788d
a320f6f14702811d a333fd156f7fd862 a3428b380ab04e3c a34efa56fb9022ba a36a604a18b55a35 a375bd9aaa79e1fa a3915288efdf15d1 a3a734ee748cd18b
a3adb74baa633a66 a3f4110b570cf377 a3f4f12ef7129413 a3f5ea6759a78eef a405c823d568000f a40ad7eaae32ca3c a40b083c2de9a84a a419d0f821c27c3b
a42c2de9364f5e65 a43749036fd4b13b a437ca8c64c29690 a4819ab3e388a811 a49e950bc2b1d4e9 a4ad2554c27dd8ec a4ce0bc2d664ad38 a4f7de6dac93548a
a50d19c977cdf5e8 a5111fb24837cb11 a5184eae4827e034 a51ff7d4534fecf1 a52f0793bd36ec1f a544101a9c5f2537 a55f7ea96d5c4646 a56963fecc02274c
a56a6d5f6dcfc910 a58217a445f0a8da a592774eb185b660 a59fe59cabb3116f a5a0566cd6fa3f8f a5afc806cd8b8229 a5b99f3392227e5f a5bb5710cc98070d
a5c56d61cc36d037 a5d167ae3de4a55f a5e1b786d425bcf8 a603c68eadc3af39 a60a6ecde765da1e a60d5fd30de90d3b a60e2f7e4968e2c0 a6298d6c268a7583
a62bc1c6006632ee a62dc088c7a1504a a647187668db256b a656cb899cd86b8a a65d2284a24385b4 a66ae23d7533678e a673e3c7ceba5ebe a681a464fbbb7b41
a69abae018ec1e00 a6c2a664e1c1f045 a6ce95d8fbc33ba2 a6e81190cfd459d1 a717e159ceebf572 a7278f0072ac0d2f a7313f0a7eaf4470 a7325dd3baa27c70
a73946686d9b6a66 a7399241ead671cf a75bd62807ca3f51 a75de04f103ba326 a76a5a644c9f2eb4 a7832eb7b283a9ab a79389647b329be6 a799e23091681da2
a79a3b62f0c08a66 a79e62cb8706c523 a7b70edee8f0a866 a7e36411e2acf53f a7ebe891a127cc3c a7fa0756b662c1ee a81abe21292a6107 a82a016a950f0a3e
a8786898f0cc657a a88f0bf115f5f53d a894810dee123c83 a898dfe71e44872e a89abf44cec1918b a8a1344bc68d4378 a8d758de8b9befc5 a8d8e4d5c727e3e8
a8dc0d76ac55f223 a8e7071883ecc31f a8ecee8e2392e4d4 a8f4c7dba3e66fab a90c98f4d10b877c a9213262f4e5b7b4 a95ceaa3a8330dfd a9715d07dfb6b0eb
a987a726ff91e478 a989c7bd840a4ed8 a991546454cc69c9 a99c77748810e986 a9a9917a1e1c549b a9ab722ff6572903 a9b2721ddc9fa4db a9c8dee2dd947bfe
a9f3d36dec7d5a79 a9ffff0ef74f9465 aa048116c7d29312 aa186dcb0a05536b aa29a7c314282d81 aa2a6c118536a256 aa567a1a055ea2f9 aa7692c1fbd2485c
aa7a1208397824f1 aa804bd3d5cda37e aa81df5f9bb8f3af aaa85346a7c6e051 aabee483d8f9fd20 aacd00652d4d0985 aade1c806336fa0a ab07869b926b846d
ab08b426b60a2808 ab1238e025465ec2 ab14316206573e38 ab2dcfcf92cc0e12 ab46ff77d43f2b04 ab4fdc107401ed88 ab57072443cc4c6d ab5f443d546abbb3
ab698dbffe8bbab5 ab6be779b78f3c42 ab6ed527d431bfaa ab715a44f2ccbf78 ab92090d9caad01f ab98d0e6921f138f abb774f2c1fa309e abea193ee5d41b14
ac0ba774d48568e6 ac2e8137760badb6 ac30d983f6472913 ac31a280aaf3d70d ac46a584b90c41b9 ac48dab8c975e98b ac51c9c88d95e9ad ac5ad47a79ed8585
ac692e2f9c1f62b1 ac6f08e6fb4eab79 ac79543537e14d42 ac8c1a0b17b7dc39 ac8fe3ba4cb31904 ac9513bb33a022e1 ac95ab8de031a49a acaf74a92aa812e5
acb9c9f2186d725c acbd22c9d9a3ae86 acc198ba27bd8f6a acec70780d766db4 ad3ffd0324bf311e ad60807dcd0220e2 ad6c2adc653733ac ad6d7d26d4ebe0a6
ad770dfb60d2d396 ad82d95f95417dc7 ad876cc632b77ecc adb37b634338d966 adcac4e6b1fa98b0 addeaff0446173cf adfb28e7718445e1 adfdeb521cd44a81
ae0764ec867e3494 ae087c767774aac1 ae13d5cce34d7b0a ae3baff7a638ca56 ae53c3252015946c ae59d45d05c0db21 ae66574f80c02935 ae92ce619efd7777
ae9496ded202b7e8 ae9b0458be54c860 aea292e9cf9e46cb aea570759d839633 aeb9a659045ded29 aed53ed4d2fbd907 aee22efeb0d12e60 aee5027c50d055a6
af05b784e94cb548 af0a5157e9181bf7 af0aed5b5f3676e7 af494badc24f4300 af8b0d6cf2313831 af9d3a908307e995 afabaec533a2af13 afbed413dc19c0e3
afcd0043baace20d afd5426efe5a73d6 afd6d5de173ada21 afd87c87288627ab affb39f8579b95c4 b043302caaec94a5 b04bf836d537b1c4 b0616a0a700ad401
b06d93b95f76bbb8 b0b386065d75cae4 b0d324ebb2b744b2 b0d9adeb00e86ef6 b0e2d543be880386 b0e2ec4ced56bf54 b1191fa23c9e13e7 b1237fb0eb3165c5
b14c69146410ae94 b14d9873c3e9f5d3 b18918d0b5db9a03 b18ae01b9befd38c b1d028701a6c296f b1d897cda5337fc7 b1e90c2e7611fc0f b208145bd62b0ad7
b20852b85eb0ee31 b2132d0663d25608 b2177a399790d139 b2248f80045c3c0c b236780f7d124a84 b237e75e9b90cd43 b24fbc0b9e33ca94 b250831eaee9c7c6
b253e66875ed9752 b25d023050eff543 b2631553c9c5c079 b290ac93cd09ff5c b2972348858602bf b2982cb9c18ad9a3 b29bd91aa782a678 b29f6ec3068953ca
b2a455a4c83ab6a6 b2a6de7f2c2689c8 b2b862951eb24965 b2bb51a93736a4f5 b2bc62c069582cf1 b2e78b77c306da56 b2ee23b5a01af1f8 b2fd63cb97b421dc
b3106cf84bea2bcc b313cd88eeefd740 b315be60acb07cca b31b7cb654ea5ca2 b32188e44e090a36 b32f72bddcdf49b8 b339cfa655268f96 b33a8aa7f5525412
b3406d5203c48cc1 b367733a7e04a467 b369611a5d0f9296 b371fad8c30919cf b374ef9914c30350 b3779b420f216056 b37e80cdecbf2f9e b389ba445577f885
b38a4aea48307f39 b39fc7d9d9e71372 b3ab38630a6301da b3be0d46f0468c7b b3d1baa44843226a b3ec65a83659e2c0 b4432cd0e0f9960f b4479d79d9e00149
b469e4113bab17b4 b4731e7a72466bcc b476e98eda83abae b482be295849862e b489d4824ace076b b48ee5b4d73c73f7 b48f4c7dde6b789b b4983058d6a4241d
b4a307b5910bebd5 b4c440cde609e40a b4cca42fe079c9d3 b4ea4cbbc0025d8a b4f1d8369baf42f1 b4f46b95684f1092 b528c8fc6006696b b52e83a92f1dc341
b5452f7ea8c415c5 b56610db00a051d9 b570571060434cff b5823010e436289c b59dc1fa79ae96f7 b5af13eac60e8943 b5bfb7176d9570cd b5c794fdba50030c
b5ca51bc80b49d96 b5e3791ef6fb09b6 b603f9f40c22e449 b61bdda4a9c2c25b b62d1998079c6f98 b62fef92b2a3ce42 b64a1a2ec4d78e91 b668e59f9687170a
b66b66d2962c022a b66d4d22d863ccfa b67368715e8377d5 b68f2777b931572d b6a56d99009d044c b6a720610901f9ff b6cddb6cd5e1a6e8 b6d479e71ecf76bb
b6e389457e1d58e0 b745f5031f9b8a01 b77014c4f08e31a9 b7813cd70fe16ed9 b79d3cb3fc8b8af6 b7a2aa51cf5a7f12 b7b6407965b899b3 b7bea10245c8a08e
b7bfd6ac5cde120c b7c18b50fc779785 b7cb71da7fa4c731 b7e52f5b1f5f7a1b b7eabcfc9a46107d b8024480573d5a61 b8339b2bdce9e5a2 b859c2aa14e9e553
b865b2624faa9a6f b8a359bbb5b7cd1a b91ac8a15fb5995a b932f4b272a8c457 b96530381e23e581 b96b57b218f7da95 b971b581956e620d b995333a6fe11859
b9dc621ca9cd89eb b9ddde04778fbc32 b9e7fb9f7b6855e4 b9ece7dc9c18c785 b9fc2e778cf8edd0 ba0f616d725e3ef2 ba272e79da5d2a7d ba27a5175f81bbe3
ba520ce2b65eb45f ba52e537c285bc19 ba533127cf99facd ba54e09f16eb8b18 ba78149933df7429 ba98e3ee8b8a4e52 bab6d01c5a03ef9c bac44330286c46bc
bac45fe5c316d0b9 bace2139fa203e18 baceb833b355816b badd6ba974af5e2b baed548a77319a4d baf13fca8dc5f620 baf58ec1f01eddbd baf66d9c52089e66
bb0f71e42108c5a1 bb1785c013cbabb7 bb3eed47e5a28621 bb40c1a6b0caf1b8 bb5339765ec2f95c bb54f278fe3355c8 bb67e78351a7f8ec bb840b8cfbb7fa51
bb9f08e9b463e794 bba02690ffaa3580 bba1ea9b0432867a bbae9df0c344bf84 bbb5ea3343b67399 bbc271075262c194 bbc5ebf005a163fe bbca00a4c4526880
bbf77240b148cf17 bc3df64b99853647 bc4327318fcc4ab1 bc48a90bab450d1b bc498e7e8a61d0ad bc68e4ccdaf25870 bc769d2de5147434 bc8609bfa5f2a266
bca88ccea60b4a2f bcae3eb6954d2915 bcb4cbb6436525e0 bcc0f1a602fac275 bccce983d3fe7026 bcd0e985722adaed bcd8364f356f02c4 bce282a810ac1088
bcfd4eea7a498970 bd33c986c65f23f5 bd538f024391379d bd64af5762184e84 bd837a301fbc93b6 bd8f21a35c32eeeb bdb1682e4226dacb bdb3e6b46709a897
bdbb5710c7d5fa8f bdc4396cc113786d bdc9194f715ea3ef bdd51a9b526edcf3 bdf0657386c83992 be056ec3b39641dc be165ea735d1ff64 be1eafe4ebb7efdf
be3b0ae06428e636 be47ec08b1e818bc be5511e2b8c85578 be5ec98b280edc56 be67e0ca1f2dd464 be8ac71240ccfa4f beacbf74f39e895d beb70050466075a1
bec7f91c6fac836b bed045509660a5ee bee55a07367b496e beeba06c5b43023c befdf6016b9daf7f befe4e1c6d3b19e3 bf2e6acec67db957 bf387250e1df60d2
bf398f7c64eec295 bf4292c9551e1b45 bf5093dcdec84134 bf59b5480ba9aa9a bf76acae028c5813 bf94314747cf2e3d bf99ce6da884a125 bfbe09235fdb5904
bfc6c2f7429c9aa2 bfd44e45c7e1ac32 bfe4b8946a602161 bfe943235be8f31d bff0638a6e6f5c7b bff9079af60dcbb5 bff9d05a133576b8 c0040c330178b1a0
c017b4e1abf297ce c023e202398e8121 c048ff2093b92e3e c056765a49ff4e5a c061fba9c5f86377 c06ed1b58abeee44 c087090e93591da3 c0a06653fef35db5
c0a56a81275b4794 c0adb400971ae256 c0be20db20d2d5f0 c0cd7ec19d36df57 c0db410feb1af712 c1117f9c37301deb c116cef0df76c4d9 c119ba71fe76442e
c1253552f4c3b062 c12c121d083ce68e c133392817f0a9cb c13c9415c27827be c14de4694068ca50 c151f39837193262 c15228ff196a46f3 c155bdb511f6237b
c1671eb665dfac81 c16f165eb5f4a68f c170af6df80d00e7 c17899d1f6c80869 c18438ed2613cf09 c1947dee7c9a7ab1 c194846cfc158358 c1a8e795eb5540ac
c1c886e95d79853e c1ca68a7ca9e051c c1d9089e9c271aa6 c1e3e4f7cbda3e95 c1ff136c12837f7a c2071bece243fab6 c20f9211c5077444 c2148eda2cc0d6e5
c22309aa18f92a89 c23810a38500d94c c24b583d5a827e08 c25d12f3af964da8 c25d93023cae0872 c2626c5f15d8d0ac c26d1e0185226ef6 c277a405d12a83df
c278f25603d73c4c c28f199e3d610266 c2a14aed7ad9e34a c2a3c45dae85bc6d c2a7671295bf44fe c2bc6a16da7e8fb9 c2e028a69f08f49b c2e2f1bb73ea4216
c2e6d6be5ba5f8d1 c303c38467332432 c30fee35245d1e20 c3202968e93e3793 c323a86a975bacdf c330d74b96671962 c3381c2c6aff687d c3599dd3239a4efc
c3641b04148f872a c3653cfeb08c852b c369859ddcf731d4 c36a27e23467f83a c3810c8cdccc79b8 c381ac7d135e92c0 c395030944095353 c39f4a7f604ca70b
c3a0fa1b8160bdc9 c3a250a634c63de4 c3a97715c57816e1 c3adb6b7e87c36bc c3b40870552c6603 c3e5799310cbfced c3f0646c82d082b2 c3f7b40978e902fd
c3f9da816c146a35 c3fa00f1edf334dc c4088131b985296f c40f1ff7120148c8 c419e7027621c403 c43976138bb76751 c4439e99df2ed4ea c46c72b9003eb073
c49b31b5f6408e45 c4a94e130f079c66 c4abdf0545d1716d c4b1fb2cf5f4d655 c4b9ab091f81c70b c4c27e5cf763f2a9 c4d1903427f39e38 c4ea5230bfd3c617
c53b24a0fa4c84d8 c54c1430ff51fb85 c5cf6fad3a81690b c5e95c98ba34ec72 c5ea665eab437a38 c5fda984674f8844 c60eed0ed3420e2b c64df801f3fca142
c6565b4bfb3a55f3 c68849a8d291b78a c68fd1629b63ecdb c6919316e5961ff6 c7168d3568b563f0 c721ef6de38511b3 c762276d0fc4a7e7 c76e2036b4b382be
c77a4e5137ffa942 c7b294a6fb4b2862 c7b77503023be38f c7c1674224b218ab c7c94b6b93d99044 c7d7401f141781f8 c806699b1ff0f364 c80d978e45f0fb40
c82f8c4fe21de0e0 c8327ff4f7ef79c7 c85fa9373c807cfc c876a1b2122497ac c8777e1b259de6c5 c88879af11eac358 c8b9da6ff90fc51f c8d993fdf787357c
c8ec94adb0473592 c8ed63212b4cdfe1 c8f46fc5b0dac81a c8ff879ce56c9ce9 c93d0fe2371e6b96 c949d0f967b55ca0 c9544084c16e81eb c961c7310f868895
c97650665c2aa0c3 c97c8c7c7b10d204 c97cbaf686e93b87 c98a499cf605f14f c98e2e6654c08510 c99a75daf260036b c9afc949ae72b1e4 c9bbcb85a8f365b4
c9ea7e5eaa0ec4e0 c9f766c5d6d1941f ca02926ea2825194 ca2dada3b4db5f77 ca38c10234cc6c8b ca3dc14d3747e678 ca502cafbccc9ca6 ca5494047e022e17
ca578263a589982a ca8c33b798e8709d ca98a94fab1f84fb cac9b0792e436b97 caf822b783a1b6f0 cb3a33e0e138724d cb414eb582512bf6 cb4cfadd11411bf4
cb6167c7b7db77cc cb63f493e4fe68c3 cb81e170fdafaa6d cb8de0324b1f7f68 cb9789cacb93b1f9 cbc5fb014b903c27 cbc7d378b6d83a6f cbd8c3107cd6dcb0
cbeec452483a0299 cbf74a3d2a4799cb cc126f26a2db7eaf cc16bc30063ffdb5 cc2b61117f300085 cc87bc4a4f678168 ccb55473735bf9e8 ccc021fe2bb6376b
cccac3c67d2533ca ccd113fe06aa59fb ccd7d0eb901ad8e3 ccdd3d2bbd7515c9 cce5fc2635aa1288 cce957c5a1ac0507 ccfaece06a621a53 cd148e03bfdf57aa
cd1eb45fff1a1e11 cd2487b917cd915f cd3144063927abfb cd42d7a1ee41cd66 cd4904b732d1c2bd cd4e596b7885ae01 cd62d1bc0bfdc2e5 cd736bf2b1b7dc2f
cd797226a81a1e15 cd7e391346155f7b cd7e9fa3f8423d6f cd8b0a813dad2daa cda66e59eb8947f1 cdd50b1598f4da32 cde523bde9e18b25 cde9582ecbab3ccf
cdeb937bfd7cede9 cdfc4dce131473c0 ce1360f913a51ada ce20d12009bc1150 ce361aca4c9dc1e4 ce475b0f5e1a2382 ce56514286808e2a ce5a06ce8eb0c2a5
ce63824c9c9ff406 ce649f2466e2e08e ce7cc19477b1c317 ce9659bef6e202b5 ce9bd78b46da6357 cea14e37e2ec1bba cea3c7aefb9fd2c4 cea3fa95e44be3e3
cea6926b2ed0bed2 ceb405c9f2d794f6 ced9fffd7dff580c cee09a60e7fe41eb cefbc07e2d172c63 cf0228174324135f cf10959708700829 cf16683be4534516
cf1f6a075aa6d26f cf2c1ee9e7e50153 cf3b0e49081ac0ec cf482bdbf0865904 cf4e05bd3454fd48 cf51574c6eeb1d7e cf52929dd1328d59 cf54a744c02e0fa7
cf8aaedc23cb94ff cfbb9ab75b9b5c26 d0011e1380e8c8b4 d0427c735d7af2f5 d07590ec19122ede d08e7270d233a190 d0a7c61cc47d8a5f d0aafc3ffd1e42e9
d0be56f6094cf587 d0e729effcb3e387 d0ece521cf35aa35 d0efd5129ec373cb d0f2e5058c010d1c d1080a6fd6833582 d117053650aabcae d11863baca5fa095
d1368ec89fccb998 d167aa1049ceec46 d18170d08cc01cce d18d1e07b2ed66a8 d18e83cabc04cae3 d1ca30c828fb77bc d1ec630369b5a803 d1ee42c0657e47c6
d1fcdc0b9d21bdf7 d20f866ade67e750 d22c02ef90898a7d d236777bc9343ead d24895393f9428a9 d258d808dde07359 d25d94526e86b764 d25e3e67d1acada6
d26a739398be6a01 d2711b0656bf4008 d29105a420f9a351 d2a07fe2d2e8530a d2ab4d89208172f1 d2aebe807204974f d2af489cc02be4b8 d2d428989579bb75
d2eca35be28590fd d3020998486b7e20 d30dc90a70efebde d31b83cf067c023a d32bbc56ef045d61 d34b1c4b3ae92e14 d3643ef9737cf8e1 d36e30d60e239966
d370dbb577ca8312 d3716d68ead51829 d37e82d921d828a9 d3841ace9ff49d3c d3e608fb979910fe d3ef758cd5265e24 d42c636ae3b5a4e9 d4449be79e3e6987
d4458ed5d193458c d44636de51e26243 d44d31ca2435f459 d451d94fdf65e22b d458476ec6d069da d45b03d6eb768e97 d45b610ef0acbdbd d46118ebb96691fa
d4774a410c0aa791 d48f4fccdcea13c5 d49c14f8732bc01b d4b68593358318f7 d4bbd42724eceec5 d5069882ce36fb15 d514c4378ce61606 d517954cebe56eab
d52ca982e548ef62 d5851210ea334715 d58e0c1746c7ecca d595c7005bd6ff3b d5a5a055e1571a72 d5ae7094298b3873 d5b39008d1ea0f29 d5b7fa0fbc3e3ff5
d5d10b12f57bb143 d5e631be7664eb30 d5f86627e623d560 d60d862989e15455 d60e7df5b96ac3df d64d5c14619fca1a d6568deccd1ef047 d6610acbfc4a66af
d674ec23166a03d7 d68cbb0d460a3878 d6a3de7eea7b749a d6bf51cd3e298824 d6c24217b9a68f4e d6d6402fdc9c8e7f d6dadef78666bb45 d6e41e892f7a0419
d6f3b01cecb462ec d70a3b382e4c8509 d712056f21809fd3 d71fe4bdc265ff53 d7262ff8f00c6d7c d760102b4376b3fe d761bdc830d9aac8 d7625b313419f6b1
d777b26b187c8462 d78204e9b0731350 d79ad73beefb94a8 d7bcb7cfe9243033 d7d497646ce5e740 d7e212df2c0fbacd d7e703fa5073aad8 d7efa24d710d28f3
d80b82559f605049 d8312282579338e0 d85094578795c4d3 d86c60c2a71d7d75 d87fce4f8676abd3 d883b5cf6a163749 d88c09a44bd43022 d896e3bc53ee6421
d8a58b3f2e99701b d8e4d3a2ad5f2ea4 d9074c2f785f6033 d908b0785f067677 d912371c1700f707 d91f683513239099 d94e39ac6135daa7 d977123e6342cfa5
d99a2f2240377586 d9b0b4318a790680 d9b50fa09c1d81a7 d9be204bb523076f d9c8ece8071ef828 d9e5980272feeac5 d9fda613b0983beb da13616bd40d939b
da1ba3888da63870 da312311ce048be0 da3ed6e871d97a1e da42e35d15c1f5c6 da48cbae90a79e6e da4c46dbb77e43a8 da500a699da7b491 da5a77a590f8ab9f
da5b17211529d329 da6e0b720a8d2ed5 da8763a4f8621775 da99ddc702a205dc da9d78bc3f86a348 daace517c813b5a7 dab379a4102e59e1 dac0a54411308f3a
dae9c32af8c7861e daf081f1f7c6cff6 db07694bc263e26b db1bf4f057f09990 db20ee6b0404b32e db421ff33025bdfd db7266bb70ec6203 dbb31cbb309f1737
dbb3277409262f25 dbc383d329306381 dbc72ba7bf24d8c5 dbd07076fe69cf49 dbdd531d45b06d52 dbfd60972e21a8f9 dc1bac347085040c dc24a27c23efe9e8
dc4bc8186e9e5d3c dc8cbfc5f54910c7 dcc8c7d95189effe dcc9f72efec7ac47 dd0f7df4312dcb1c dd12fae59b71d092 dd2777dbc5985ec9 dd2eda94d19893b1
dd37d0580d84d6eb dd3ffae74e9d5438 dd468a14c252ce22 dd5a5d1c76d6acb6 dd64d71199a9f4b3 dda437505925a636 dda4c55042b20c47 ddaf3a0b9d33fa4a
ddb2cf36a1d639a4 ddb4a8c5b2e7a322 ddba481cf166f743 ddd70141ecc0f57a dde649044c961610 ddecbd508477f7d2 ddf4b5dfe53ceaf0 ddfe50c26cd08f44
de0e599f8db8adbe de1988a987c86456 de1b76d3ec5153fa de2090bb4f4919f6 de2ff7157e3c2468 de4ec4bdae247530 de52caed830117e5 de5f54113fc92668
de780ab1748befc4 de7b0386b491c11c de8a3eb279bd47f4 de924d613fe2f409 debb364f212f58dd deca305ccedfd281 ded2fedaf11a058a defad33ff631c62c
defebd1637c124fc df09b4e2c715b84b df123eb4078aa7e3 df183b22fac480cb df39f6d66c1f5b81 df5e749bef410364 df68479aca5513a0 df865339026bc36d
df8e9362d8625e2a dfa8184f9fead55e dfa83d7b7aea94bc dfb10907836506b6 dfd9f0b57d983312 dfe3e444e9d5d015 dff7d0fc43984fc6 e0021b6bfffdc939
e00f2e9105ecb73e e02301cbfd23cdd5 e023610ba9cf6e5d e023e5b6cd37db0e e025532f092e2ccb e0375b3d8eb0660d e043cadcbee79297 e04d4be4f68dea72
e0503b51d72d8c9e e0503caa3ce8d20e e05702213d9fdfac e05d1e36992d3d6c e06273b1244714c7 e06db92bab6d4a83 e08deab4d2a1f209 e0946f5285626530
e0de16e7e61cf0e2 e0f03ff2f1dd0ff0 e124c94156212b82 e14ac1fb9a70e62d e168356983327f5d e16ef041fef16266 e18b432fa99d32b6 e196d030a59d7890
e19889d9d4ff941b e19c1c898ab870bd e1a3f1d368cfacd9 e1b1f31556cb5c95 e1d8b20ef197ed2e e1ea6a08037710f9 e20585c29090b6bf e214ec849b4d9874
e2243f15370914c0 e24e19def9a35ffb e2539fc9a53244e6 e266df90b8508309 e26db417a68ac5df e293d0c6b55fe2db e2944a5f40083c88 e29fa52f4ec47ff1
e2aa64776f04032d e2ba1d8206c4648b e2f92378ca2caf12 e306a20e715b6947 e3166bc6b325e6b7 e318623f2881fd67 e31d469488888e4b e327578c4a03be48
e32bb59dfde0a80e e3307841c636368e e33648eb47508584 e34b2debb47f24a2 e350b1c66d47966c e3529623bb414183 e3541bc11dd89c06 e35e564c5569d7ad
e35fda3ff41972bf e360696e3b361532 e38c30ba1f46bdf8 e391e2664b2b7436 e3d30f1063480388 e40cc257ca00b5e7 e44c6f0b785a7015 e46409c9a055d6d6
e47239ac7e744b58 e472552cf95953e7 e48559d93afbf026 e4976d51ce8a0049 e49db32b5397d76e e4a140a41cc40a5e e4a2e7f3d8aa63e8 e4b54acf5fd2cdbf
e4cc9a39a0a38342 e4d347c1bd0bb6de e4e860168794e38e e4fa1798a7055f73 e4faa87862c975cb e4fee1dfa1cdab87 e500e8891c836a6e e507a0ed5910ed7e
e507e7108710e546 e5096105b4a02514 e53d1bdd3e05b81a e54396c57979dc02 e5590c5e831f7971 e56b49a9f8ae850f e5719f98f8c9ced2 e57615c4717f55b1
e5b27a74172c82d7 e5c3dc6aab8df3d2 e5ccab2307b12281 e5d774f55010b633 e5e41ecdb4bcbd53 e5fb3beda5afbdf9 e5fdefd07c785cd4 e6036c01e3aef466
e6074f248cc5d4bf e620d91c11b05c55 e6324faea8ba24f3 e63af602e1d025ec e6537268e158042e e65e70679c188ba9 e6874a670a7d11e9 e6c0f0c2a5cfe409
e6c91cb606006a2e e6e5f1c01da98258 e6e7fc3935e8cf1f e6ff4b592bf9ea64 e710d0d21a0b5fd5 e724fdcfcc4051d6 e725a1e64243a5a5 e734f1d35c3ae380
e742fa5137cc06bb e779b20662d4b57e e7ad4f8c828931b4 e7b732586b6b8642 e7bbb622668f6bc8 e7beef7621f56e2d e7d07eb70494b459 e7d6be73a8384d0e
e7d6ef166b8c8adb e83d5ae0fbfbbd80 e851162f89101f19 e85e8ca4efd2fa4b e863cae64cd6cf43 e86ad03d8c68f763 e87a7814c5c7507c e8ab5af29a8116ab
e8c4050212213e3c e8c83d0d5b90c95b e8ca081497033379 e8de8cb4ebd5b766 e9122379edb05db7 e914a27844b969a3 e92aa035702bfff2 e934dca456be14fc
e93584dfeb549029 e94bc125f6364650 e9522e422600e32b e95356afa3f4d401 e95e8bb55494b40d e98d4f54beb13d81 e9a12910641c779c e9a5058703ad58a9
e9aebe9a15501659 e9b78350bbfa3183 e9ba64b868d1f2a1 e9c1f92fd90f87a9 e9c64f346053b3a1 e9e6b1381ac98344 e9ec3c69544bff0b e9f0f3d9eb739574
e9f2e857901fc71a e9fcee75fe04b1f3 e9fd397b4f0697c0 e9fe58a95f264419 ea0dfc16e0fb41aa ea1b4df481ff094d ea217c253d160c27 ea362b9f0b1aa6c6
ea60b5d0862218ff ea6eb363fa5c076f ea7d442a583fd8f1 ea85d827b643b8fa ea8cc5773b5e8eec ea97c4c89a58640f eab31d427b124721 eac78672cad4cf96
eacc4c2ddb0d2a18 ead90913bb80791b eade9d941cee6a70 eaeb05e0a4fddae9 eaf238fe90035ae8 eaf3d60e6c468ebe eaf8091fe2742bf1 eb0169043bd5456e
eb11802616746fa6 eb1d502ce81bb699 eb32341f9ccba478 eb4c1b8ce13a0d8c eb4c56e00aba9c6c eb55f01d752800b4 eb8ae8e4e2c0ef17 eba22d045cb3647a
eba9423902df8e75 ebb43447c58d05da ebda6190757a0548 ebe33f0a2e047c5d ec00db7ee1446f8f ec2eea7a6042eba9 ec3f275ff0a194e2 ec4a0a9cd7691e99
ec4b06b5f326a3b2 ec4f704af39fa06e ec59db98e8f9f4f6 ec5df16accfb9829 ec5fe61972385bc3 ec603cd9c79382d1 ec654001ca2dab07 ec706aa66a9a2962
ec75794a184328d6 ec82ba37998af4c8 eca3b3612ff54c5f ecdd8a10f9610156 ed35d6b980b927c0 ed4308f879a8229a ed6c46e15caa135a ed83ed0849b8e0f4
ed9cbbf4bc3d1090 edb9ec1f8f68cac3 edc1147dedabdcb1 edc13726d8e654e8 edd3fb7d708764cd edea64752e946998 edf08e81861a7f73 edf1908dcbbd77fd
edfbc378d17eb3ad ee16cf0a5f85e7a3 ee21ae015c0be85e ee3ebc02c3833045 ee424e586c7fbcbc ee43b5132e51516b ee79cb998d02f6e3 ee7b5057688da125
ee84bcdb53da3672 ee9a9c3066a70fac eecf9a97ce624060 eecffc2f7f5161a8 eed0a9c96f00487d eed1fee5b6a07270 eee6a884fe263ed6 ef099a100c7fe9f2
ef12ff8b8f572fbb ef1e4e91b22226d0 ef41da2bc0ad4505 ef4dc39ee7dac682 ef4e13521447c881 ef54231615f10256 ef559e494c562641 ef6b4f762be640bb
ef7005cfa74d8682 ef91e79bc6b79d2f efa24351de0dc96a efa9cae6e97b86ec efb4df8eb9812649 efb8093a8f11718a efbdab76688eecfd efd4757f0ac6d585
efd4971291433e1f efea52f74491120c f01f2c4d8616ac42 f0540b0123e60e92 f05b395898c974cd f09ef1dd9f73808d f0b1089511c38ceb f0d4fd57e15bcc11
f101d304e628cbb1 f106f911de580bc4 f10e51f7ff6be051 f11170058421c518 f1161babe09334f6 f117c512c3d6045f f12b8d20c59e41f1 f12dafb5151ddc2f
f1305123ef6fecea f154a349c2613845 f17df6924d4180a3 f182de3b675e7f3f f18b14cb33e805b2 f1921bcc75b23270 f19f8b75aaa33475 f1aed60525fb4302
f1af7a1e3f5e3cd3 f1bb6ec529edb8a9 f211dd37671c1c43 f2188b92771c9967 f21dca42eff424a1 f22971b306f82f8f f23e8d4f33ba813f f24d3ad541bc9909
f25356953d299ea1 f25409e4edda287f f29383c8b9066abe f2964555f2bb9b09 f2a405481fddf30c f2a74c6860b759c7 f2b36d10f0142608 f2cb9c098775cb00
f30c68268aff9622 f313ff6a7a49a375 f31d306bbf6d8d5b f342cd42a6a13d85 f3480aaeddcf97e6 f35b5a40311dff1a f35f178499e71d35 f3607c169ebdf812
f396e85b684941c1 f3a82bb5dbca4263 f3bb04669dc970bf f3caeed15927ef0e f40003de50c2e45a f40be06220cca8e3 f40ddb1f2d107912 f4183e8aa99a41cb
f41af9fa3a9ee3b1 f426d1fc03f2842d f427538eef0a3978 f42c1ffb855c7298 f4317d098af5c381 f485a69dd64be087 f49db01ce19883e7 f4a5afd85eac6726
f4a8d9ab56f66163 f4af72ec68031c28 f4bdcd6454e99077 f4d5168c35ba762f f4eb537e924113e8 f4ecb0fd1b115a47 f4f14160d84d6bb2 f4f9c54e0c973ad1
f50c283d80fe8559 f50f2a1da8a0ee2c f518ae72a0e664e3 f539133034aaf6bf f53973e3183c6377 f5573e1be5f18387 f557d576db3be166 f56e3530005f1b83
f585f4e286d26281 f5c4a9eedd34a212 f5d40dc86cf6367f f5d6794681204cc3 f5f20f47776a47e9 f5f2236a45920d92 f6044ea8fa354371 f60d1ef038b7751e
f6233b3c10a03e93 f62863cd9ba17095 f62ffa569029cc38 f638a69a7093c1a0 f66f172eea580028 f679ebe78078dec1 f67ad149413974b0 f6825bc21a6f9526
f694bca357ce1694 f69a907e73c394c9 f6bbede378348766 f6bd6595dbcafcf2 f6bf99f849a88845 f6c5e70c60154e8b f6e66bd536a0b7b9 f6ea38156eb1e1f1
f6eaa491651ca791 f6ed0665a33a4ebe f6fc1b4f62012d6b f719fe2d429aedbe f7337cd64e25daa0 f7437a8cb36c1f91 f74759ee501736c5 f7550e72e95fc651
f761bfde4004e40d f77e1c4f53facbe0 f7851df778186df4 f7906dc1449fdbcf f7924f255ed47b95 f7a63566365ebd86 f7ac79e17b25d67c f7aecbb294554806
f7b2110bb000732b f7bb630d4f14f28d f7cd148937fda0d3 f7d2fad12403c194 f7d33a43ca9954a4 f7e51285967e4625 f816253c311987a5 f820a56b70942cbc
f8229c0cf41b5517 f8288b403428b579 f83ba53c640e90bf f83ce9dbb7f836ba f850286ec713cdd6 f850660297d366c3 f875bec2005906fc f8873f24fd035248
f897906fe3c9e09e f89a9db935577c0e f8a9216237fa3c70 f8d920493108b0f9 f8d9282cdafade19 f8df7e2a77610cd6 f918812678d700fc f92e940d05c50cb3
f934c962d371d5d8 f979ee4159654782 f97f6cea7a626b31 f98c24c4273184aa f998f2e8179247d3 f9a67c8dc3b8b32e f9ae3a6d29ab4e04 f9eaaeda138eba01
f9ef133dc9ca5561 fa116f07e76d275b fa122a4a7e420e14 fa16a21f0b5f6d42 fa1b42ec89f3750d fa220c14fee98d5d fa25aa908e6fe151 fa267a967f276dc9
fa364799c91850de fa501f0987f8430b fa50cc67b2fc4b97 fa53a05b3ae35543 fa6c8530edbaf119 fa8a2f51ec52fd4c fa911f9f66631429 fad615026bd66bde
fadbc7e884602f92 fb0cc41f10f9d6aa fb113b2ca74d4455 fb3431a4a18ff4f9 fb396b441732f13f fb57aa073d0c5726 fb5b4e48f4c236be fb67bce1c37a9316
fb74c3fd3a4e9d7f fb768d11b10d73bd fb7c3c11dce7fdd5 fb7f631f694b5277 fb81ab106132ef48 fbb1024b89f609c1 fbc23fca3fedceb9 fbc95e7f55692af9
fbe676cfbdc14399 fbedb9364fdf21bc fbfc354e6bf763f1 fc30a840c9659481 fc40bbd49e593462 fc47684912c3cfeb fc8009d209b47a60 fc80343c27c63b28
fc91eacb22127dfe fc9788b9e12fc118 fca42197fac9b9e3 fcaa0877594161e4 fcc4ff5bb1f3217d fcc69d75fc6dbb1a fcf0fda29b9d3b8b fcff20e743f59a4d
fd0d827d53d47a0e fd1e0400c6d8dceb fd36583ece75999c fd4e9a0deea85807 fd57319373067cec fd5cf7026f1ba94c fd7a766adc04c9c7 fd7f264d7af6dd8e
fd9d659d3f1e42d2 fda22e2e023656f6 fdbeafe481bc2da4 fdbf823fae0d74e5 fdc4bd376879e31c fdcd3ef1bc39336b fddf2f4e7cd6f87d fde6957d6f250479
fded686955d9abe8 fdfc7ac7e5cf3379 fe0cee340abfb321 fe0e624b5ef50b75 fe25d0c907212850 fe4a3f5bf0e9841a fe51636e23a3691f fe6f4026b70bb734
febeffb173b74937 fec7219484495809 fecd93f088186579 fed09bae1e4797b1 feec95aedaa7e81b ff213f201489d36d ff2c097b733d3e5f ff2dcea5b8000981
ff3b0dec129e10e6 ff4a8fbeab5860ad ff4d048ce307af16 ff5208f140d4f04a ff59e71d46b468e8 ff6037f2b439d179 ff733134e88cef50 ff7d2841d6638164
ff8ecfb3fbdc1b8c ff9710c47edfaf53 ffa3acd2fe9f70da ffbf45b4261130f6 fff3e5aeb0abe701
""".split()
assert len(EXCLUDED_PREFIX16) == 3749 and len(set(EXCLUDED_PREFIX16)) == 3749
f1 = pd.read_csv(FOLDS_V1)
fv2 = f1[~f1.unit_id.str[:16].isin(set(EXCLUDED_PREFIX16))].reset_index(drop=True)
h = hashlib.sha256('\n'.join(sorted(fv2.unit_id)).encode()).hexdigest()
assert len(fv2) == 14945 and h == 'c76fe7c45d4c3403b32caaa76881da33288e2ec015e0382119395a20228dc88b', 'Danh sách hàm MVD v2 không khớp'
FOLDS = '/kaggle/working/groups_and_folds_v2.csv'
fv2.to_csv(FOLDS, index=False)
print('MVD v2:', len(fv2), 'hàm')

## 4. Dữ liệu CVEfixes và kiểm tra
Kiểm tra mã băm của hai file fold, và kiểm tra lại từng hàm: `unit_id` phải đúng bằng SHA-256 của mã hàm sau chuẩn hóa.

In [ ]:
def find(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    assert hits, f'Không thấy {name}: Add Input → dataset cvefixes-v2-inputs'
    return hits[0]
UNITS = find('units_v1.parquet')
CVF_MAIN = find('groups_and_folds_cvefixes_v2_len768_paired.csv')
CVF_V2 = find('groups_and_folds_cvefixes_v2.csv')
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
assert sha(CVF_MAIN) == '9938574e4059f0471663b84f177c9671b75be686ee34b098bc44f8e5365b7d9f', 'File fold tập chính không khớp bản đã kiểm tra'
assert sha(CVF_V2) == '7b41d114a580b054db807a6f419d6625b9c5c5750ab36c1f11cd40332bd958e4', 'File fold v2 không khớp bản đã kiểm tra'
u = pd.read_parquet(UNITS, columns=['unit_id', 'func', 'language', 'target', 'cvefixes_only_eval'])
fm, fa = pd.read_csv(CVF_MAIN), pd.read_csv(CVF_V2)
assert len(u) == 46411 and len(fm) == 19907 and len(fa) == 36748
sub = u[u.unit_id.isin(fa.unit_id)]
assert len(sub) == 36748, 'units_v1.parquet thiếu hàm'
norm = lambda s: s.replace('\r\n', '\n').replace('\r', '\n').strip()
bad = int((sub.func.map(lambda s: hashlib.sha256(norm(s).encode('utf-8')).hexdigest()) != sub.unit_id).sum())
assert bad == 0, f'{bad} hàm không khớp unit_id'
chk = u.set_index('unit_id').loc[fm.unit_id]
assert (chk.language.values == fm.language.values).all() and (chk.target.values == fm.target.values).all()
assert int(chk.cvefixes_only_eval.astype(bool).sum()) == 12982
print('CVEfixes OK: tập chính', len(fm), '| v2', len(fa), '| tỷ lệ nhãn 1 tập chính', round(fm.target.mean(), 3))
del u, sub, chk

## 5. Ghi mã vào repo
`grouped_cv_splits.py` giữ nguyên như các lần trước; `run_cvefixes.py` là runner cho CVEfixes (xem mô tả ở đầu file).

In [ ]:
%%writefile /kaggle/working/repo/src/training/grouped_cv_splits.py
"""
Chọn dữ liệu cho Grouped 5-fold CV (v1). Chỉ dùng pandas, không cần GPU.

Đầu vào:
  clean  : DataFrame từ MVDDatasetLoader().clean_exact_hash() (18.694 đơn vị) có cột unit_id, func, language, target.
  folds  : groups_and_folds_v1.csv (cột unit_id, group, is_giant, fold, inner_val_fold0..4).

Quy tắc (mọi phép chia đều theo CỤM `group` = thành phần liên thông của CVE, xuyên ngôn ngữ):
  mono(L, k)  : test = đơn vị của L ở fold k; val = đơn vị của L ở cụm có cờ inner_val_fold{k};
                train = đơn vị của L ở fold != k và không phải val.
  multi(k)    : như mono nhưng gộp cả 7 ngôn ngữ.
  lolo(T)     : test = TOÀN BỘ đơn vị của ngôn ngữ T (không cần chia). Nguồn = 6 ngôn ngữ còn lại,
                bỏ mọi đơn vị nguồn cùng cụm với một đơn vị của T (khử nhiễm theo CVE);
                val = đơn vị nguồn có cờ inner_val_fold0 (các cụm ngoài fold 0, ~9%); train = phần còn lại.
Mọi hàm đều kiểm tra: không cụm nào xuất hiện ở hai tập khác nhau.
"""
from __future__ import annotations

import pandas as pd

LANGUAGES = ["C", "C#", "C++", "Go", "Java", "JavaScript", "Python"]
N_FOLDS = 5


def attach_folds(clean: pd.DataFrame, folds: pd.DataFrame) -> pd.DataFrame:
    need_clean = {"unit_id", "func", "language", "target"}
    need_folds = {"unit_id", "group", "is_giant", "fold"} | {f"inner_val_fold{k}" for k in range(N_FOLDS)}
    if need_clean - set(clean.columns):
        raise ValueError(f"clean thiếu cột: {sorted(need_clean - set(clean.columns))}")
    if need_folds - set(folds.columns):
        raise ValueError(f"folds thiếu cột: {sorted(need_folds - set(folds.columns))}")
    if clean["unit_id"].duplicated().any() or folds["unit_id"].duplicated().any():
        raise ValueError("unit_id bị trùng")
    cols = ["unit_id", "group", "is_giant", "fold"] + [f"inner_val_fold{k}" for k in range(N_FOLDS)]
    df = clean.merge(folds[cols], on="unit_id", how="inner", validate="one_to_one")
    if len(df) != len(clean) or len(df) != len(folds):
        raise ValueError(f"unit_id không khớp: clean={len(clean)}, folds={len(folds)}, merged={len(df)}")
    # nhãn và ngôn ngữ trong file fold phải khớp với dữ liệu sạch
    chk = folds.set_index("unit_id")
    for col in ("language", "target"):
        if col in chk.columns:
            a = df.set_index("unit_id")[col]
            b = chk[col].reindex(a.index)
            if not (a.astype(str) == b.astype(str)).all():
                raise ValueError(f"cột {col} lệch giữa clean và folds")
    for k in range(N_FOLDS):
        df[f"inner_val_fold{k}"] = df[f"inner_val_fold{k}"].astype(bool)
    df["is_giant"] = df["is_giant"].astype(bool)
    return df.reset_index(drop=True)


def _assert_disjoint_groups(**parts: pd.DataFrame) -> None:
    names = list(parts)
    groups = {n: set(parts[n]["group"]) for n in names}
    units = {n: set(parts[n]["unit_id"]) for n in names}
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            inter = groups[a] & groups[b]
            if inter:
                raise AssertionError(f"RÒ RỈ: {len(inter)} cụm xuất hiện ở cả {a} và {b}")
            if units[a] & units[b]:
                raise AssertionError(f"RÒ RỈ: unit_id xuất hiện ở cả {a} và {b}")


def split_mono(df: pd.DataFrame, language: str, k: int):
    d = df[df["language"] == language]
    val_flag = d[f"inner_val_fold{k}"]
    test = d[d["fold"] == k]
    val = d[(d["fold"] != k) & val_flag]
    train = d[(d["fold"] != k) & ~val_flag]
    _assert_disjoint_groups(train=train, val=val, test=test)
    return train.copy(), val.copy(), test.copy()


def split_multi(df: pd.DataFrame, k: int):
    val_flag = df[f"inner_val_fold{k}"]
    test = df[df["fold"] == k]
    val = df[(df["fold"] != k) & val_flag]
    train = df[(df["fold"] != k) & ~val_flag]
    _assert_disjoint_groups(train=train, val=val, test=test)
    return train.copy(), val.copy(), test.copy()


def split_lolo(df: pd.DataFrame, target_language: str):
    tgt = df[df["language"] == target_language]
    src = df[df["language"] != target_language]
    contaminated = src["group"].isin(set(tgt["group"]))
    n_removed = int(contaminated.sum())
    src = src[~contaminated]
    val_flag = src["inner_val_fold0"]
    val = src[val_flag]
    train = src[~val_flag]
    _assert_disjoint_groups(train=train, val=val, test=tgt)
    assert (train["language"] != target_language).all() and (val["language"] != target_language).all()
    return train.copy(), val.copy(), tgt.copy(), n_removed


def build_jobs(df: pd.DataFrame, modes=("mono", "multi", "lolo"), languages=None, folds=None):
    """Danh sách job (mode, key, fold) theo thứ tự chạy: multi/mono theo fold rồi ngôn ngữ, lolo theo ngôn ngữ."""
    languages = list(languages or LANGUAGES)
    folds = list(range(N_FOLDS)) if folds is None else list(folds)
    jobs = []
    if "mono" in modes:
        for k in folds:
            for lang in languages:
                jobs.append(("mono", lang, k))
    if "multi" in modes:
        for k in folds:
            jobs.append(("multi", "ALL", k))
    if "lolo" in modes:
        for lang in languages:
            jobs.append(("lolo", lang, -1))
    return jobs


def select_jobs(all_jobs, specs):
    """Chọn job theo danh sách chuỗi 'mode:key:fold' (vd 'mono:C#:0', 'multi:ALL:2', 'lolo:Java:-1'), giữ thứ tự đã cho."""
    index = {(m, key, k) for m, key, k in all_jobs}
    out = []
    for s in specs:
        parts = s.split(":")
        if len(parts) != 3:
            raise ValueError(f"job sai định dạng (cần mode:key:fold): {s!r}")
        job = (parts[0], parts[1], int(parts[2]))
        if job not in index:
            raise ValueError(f"job không tồn tại: {s!r}")
        if job in out:
            raise ValueError(f"job bị lặp: {s!r}")
        out.append(job)
    return out


def get_split(df: pd.DataFrame, mode: str, key: str, k: int):
    if mode == "mono":
        return (*split_mono(df, key, k), 0)
    if mode == "multi":
        return (*split_multi(df, k), 0)
    if mode == "lolo":
        return split_lolo(df, key)
    raise ValueError(mode)


def plan_table(df: pd.DataFrame, epochs: int = 5, jobs=None) -> pd.DataFrame:
    rows = []
    for mode, key, k in (jobs or build_jobs(df)):
        tr, va, te, removed = get_split(df, mode, key, k)
        rows.append(dict(mode=mode, key=key, fold=k, n_train=len(tr), n_val=len(va), n_test=len(te),
                         pos_train=float(tr["target"].mean()), removed_for_decontamination=removed,
                         sample_passes=len(tr) * epochs))
    return pd.DataFrame(rows)


In [ ]:
%%writefile /kaggle/working/repo/src/training/run_cvefixes.py
"""
Runner cho CVEfixes (bản bổ sung lần 4 của osf.io/96a2g). Dẫn xuất từ run_grouped_cv_v2.py; phần huấn luyện,
cấu hình mô hình, cách chọn checkpoint (AUROC validation macro theo ngôn ngữ) giữ nguyên.

Hai loại job (dạng "EPOCHS:mode:key:fold"):
  1. CV trên CVEfixes (tập chính v2_len768_paired, 19.907 hàm): mono/multi/lolo như MVD, chia theo cụm CVE bằng
     grouped_cv_splits.py. Sau mỗi epoch dự đoán validation và test (để vẽ đường học, không dùng để chọn gì).
     Với mono/multi: dự đoán THÊM các hàm của v2 đầy đủ không thuộc tập chính (hàm không ghép cặp, hàm dài hơn
     768 token) cùng fold (và cùng ngôn ngữ với mono), bằng trọng số tại epoch được chọn -> extra_predictions.csv.
     Các hàm này thuộc cụm của fold k nên không bao giờ có mặt trong tập huấn luyện của job fold k.
  2. Huấn luyện trên MVD, kiểm tra trên CVEfixes: xmulti:ALL:-1 và xmono:<ngôn ngữ>:-1.
     Train = hàm MVD v2 (của ngôn ngữ đó với xmono) không có cờ inner_val_fold0; validation = hàm có cờ đó (theo cụm).
     Test = hàm của tập chính CVEfixes có cvefixes_only_eval (CVE không có trong MVD, hàm không trùng MVD).
     Test chỉ dự đoán một lần bằng trọng số tại epoch được chọn.
Hai bộ dữ liệu dùng hai bộ nhớ đệm mã hoá riêng (cùng unit_id có thể mang nhãn khác nhau giữa hai bộ).
"""
from __future__ import annotations

import argparse
import gc
import json
import sys
import time
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from torch import nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

from src.preprocessing.dataset_loader import MVDDatasetLoader
from src.preprocessing.tokenization import UniXcoderInputPipeline
from src.training import grouped_cv_splits as S
from src.training.modeling import UniXcoderBinaryClassifier
from src.training.training_engine import (
    EncodedPilotDataset,
    PilotBatchCollator,
    build_train_loader,
    build_validation_loader,
    optimizer_step_count,
    preencode_frame,
    set_global_seed,
    train_one_epoch,
    warmup_step_count,
)

CONFIG_PATH = REPO_ROOT / "configs" / "rq1_mono_training_v1.json"
CHECKPOINT_METRIC = "val_auroc_macro_by_language"


def env_info() -> dict:
    import transformers
    return dict(gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                torch=torch.__version__, transformers=transformers.__version__)


def clean_key(key: str) -> str:
    return key.replace("#", "sharp").replace("+", "plus").lower()


def job_dir(out: Path, mode: str, key: str, k: int, seed: int, epochs: int | None = None) -> Path:
    if epochs is not None:
        out = out / f"e{epochs}"
    return out / mode / clean_key(key) / (f"fold{k}" if k >= 0 else "all") / f"seed_{seed}"


def job_done(d: Path) -> bool:
    return (d / "predictions.csv").is_file() and (d / "log.json").is_file()


class RecordCache:
    """Mã hoá từng đơn vị đúng một lần cho cả phiên chạy."""

    def __init__(self, pipeline: UniXcoderInputPipeline) -> None:
        self.pipeline = pipeline
        self.records: dict[str, object] = {}

    def dataset(self, frame: pd.DataFrame) -> EncodedPilotDataset:
        missing = frame[~frame["unit_id"].isin(self.records)]
        if len(missing):
            ds = preencode_frame(missing[["unit_id", "func", "language", "target"]], self.pipeline)
            for i in range(len(ds)):
                rec = ds[i]
                self.records[rec.unit_id] = rec
        return EncodedPilotDataset([self.records[u] for u in frame["unit_id"]])


@torch.inference_mode()
def predict(model: nn.Module, loader, device: torch.device, criterion: nn.Module):
    model.eval()
    ids, langs, ys, ps = [], [], [], []
    loss_sum, n = 0.0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss_sum += float(criterion(logits, labels).item()) * int(labels.shape[0])
        n += int(labels.shape[0])
        ps.extend(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().tolist())
        ys.extend(labels.cpu().tolist())
        ids.extend(batch["unit_ids"])
        langs.extend(batch["languages"])
    return pd.DataFrame({"unit_id": ids, "language": langs, "true_label": ys, "prob_vulnerable": ps}), loss_sum / max(n, 1)


def macro_auroc(frame: pd.DataFrame) -> float:
    vals = []
    for _, g in frame.groupby("language"):
        if g["true_label"].nunique() == 2:
            vals.append(roc_auc_score(g["true_label"], g["prob_vulnerable"]))
    return float(np.mean(vals)) if vals else float("nan")


def build_model(cfg, device):
    mcfg = cfg["model"]
    return UniXcoderBinaryClassifier(
        model_name=mcfg["name"], revision=mcfg["revision"],
        expected_hidden_size=int(mcfg["classification_head"]["hidden_size"]),
        num_labels=int(mcfg["classification_head"]["num_labels"]),
    ).to(device)


def run_job(mode, key, k, seed, parts, caches, cfg, device, out: Path, epochs: int, per_epoch_test: bool) -> dict:
    """parts: dict(train, val, test, extra|None, removed); caches: dict(train, test) — RecordCache cho từng bộ dữ liệu."""
    d = job_dir(out, mode, key, k, seed, epochs)
    tr, va, te, ex_units, removed = parts["train"], parts["val"], parts["test"], parts.get("extra"), parts["removed"]
    ex = cfg["training_execution"]
    bsz = int(ex["physical_batch_size"])
    max_epochs = int(epochs)
    accum, clip = int(ex["gradient_accumulation_steps"]), float(ex["gradient_clipping_max_norm"])

    set_global_seed(seed)
    collator = PilotBatchCollator(caches["train"].pipeline)
    train_loader = build_train_loader(caches["train"].dataset(tr), collator, bsz, seed=seed)
    val_loader = build_validation_loader(caches["train"].dataset(va), collator, bsz)
    test_loader = build_validation_loader(caches["test"].dataset(te), collator, bsz)
    need_state = (not per_epoch_test) or (ex_units is not None and len(ex_units) > 0)

    model = build_model(cfg, device)
    criterion = nn.CrossEntropyLoss()
    o = cfg["optimizer"]
    optimizer = AdamW(model.parameters(), lr=float(o["learning_rate"]),
                      weight_decay=float(o["weight_decay"]), eps=float(o["adam_epsilon"]))
    total = optimizer_step_count(len(train_loader), accum, max_epochs)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_step_count(total, float(ex["warmup_ratio"])), num_training_steps=total)

    t0 = time.time()
    best_score, best_epoch, log = -float("inf"), -1, []
    vby, tby, best_tpred, best_state = {}, {}, None, None
    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device, accum, clip)
        vpred, vloss = predict(model, val_loader, device, criterion)
        vby[f"p_e{epoch}"] = vpred.set_index("unit_id")["prob_vulnerable"]
        if per_epoch_test:  # chỉ để mô tả quá trình học, KHÔNG dùng để chọn checkpoint
            tpred_e, _ = predict(model, test_loader, device, criterion)
            tby[f"p_e{epoch}"] = tpred_e.set_index("unit_id")["prob_vulnerable"]
        auroc = macro_auroc(vpred)
        score = auroc if not np.isnan(auroc) else -vloss
        f1 = float(((vpred.prob_vulnerable >= 0.5) & (vpred.true_label == 1)).sum() * 2 /
                   max(((vpred.prob_vulnerable >= 0.5).sum() + (vpred.true_label == 1).sum()), 1))
        log.append(dict(epoch=epoch, train_loss=float(train_loss), val_loss=float(vloss),
                        val_auroc_macro=None if np.isnan(auroc) else float(auroc), val_f1_at_0p5=f1))
        print(f"[{mode}/{key}/fold{k}/seed{seed}] epoch {epoch}/{max_epochs} train_loss={train_loss:.4f} "
              f"val_loss={vloss:.4f} val_auroc={auroc:.4f}", flush=True)
        if score > best_score:  # nghiêm ngặt: hoà thì giữ epoch sớm hơn
            best_score, best_epoch = score, epoch
            if per_epoch_test:
                best_tpred = tpred_e
            if need_state:
                best_state = {n: t.detach().to("cpu", copy=True) for n, t in model.state_dict().items()}

    if need_state:
        model.load_state_dict(best_state)
    if per_epoch_test:
        tpred = best_tpred
        if need_state:  # kiểm tra trọng số nạp lại đúng là của epoch được chọn
            chk = te.head(min(64, len(te)))
            cpred, _ = predict(model, build_validation_loader(caches["test"].dataset(chk), collator, bsz), device, criterion)
            ref = best_tpred.set_index("unit_id")["prob_vulnerable"].reindex(cpred["unit_id"]).to_numpy()
            assert np.allclose(cpred["prob_vulnerable"].to_numpy(), ref, atol=2e-3), "trọng số nạp lại không khớp epoch được chọn"
    else:
        tpred, _ = predict(model, test_loader, device, criterion)
    meta = te.set_index("unit_id")[["group", "fold"]]
    tpred = tpred.join(meta, on="unit_id")
    tpred["mode"], tpred["seed"] = mode, seed
    d.mkdir(parents=True, exist_ok=True)
    tpred.to_csv(d / "predictions.csv", index=False)
    n_extra = 0
    if ex_units is not None and len(ex_units) > 0:
        xpred, _ = predict(model, build_validation_loader(caches["test"].dataset(ex_units), collator, bsz), device, criterion)
        xpred = xpred.join(ex_units.set_index("unit_id")[["group", "fold"]], on="unit_id")
        xpred["mode"], xpred["seed"] = mode, seed
        xpred.to_csv(d / "extra_predictions.csv", index=False)
        n_extra = len(xpred)
    pd.concat([vpred.set_index("unit_id")[["language", "true_label"]], pd.DataFrame(vby)], axis=1) \
      .reset_index().to_csv(d / "val_by_epoch.csv", index=False)
    if per_epoch_test:
        pd.concat([tpred_e.set_index("unit_id")[["language", "true_label"]], pd.DataFrame(tby)], axis=1) \
          .reset_index().to_csv(d / "test_by_epoch.csv", index=False)
    summary = dict(mode=mode, key=key, outer_fold=k, seed=seed, n_train=len(tr), n_val=len(va), n_test=len(te),
                   n_extra=n_extra, removed_for_decontamination=removed, selected_epoch=best_epoch,
                   max_epochs=max_epochs, per_epoch_test=per_epoch_test,
                   checkpoint_metric=CHECKPOINT_METRIC, best_val_score=float(best_score),
                   seconds=round(time.time() - t0, 1), env=env_info(), epochs=log)
    (d / "log.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    del model, optimizer, scheduler, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


class Data:
    """Nạp CVEfixes (bắt buộc) và MVD v2 (chỉ khi có job huấn luyện trên MVD)."""

    def __init__(self, units: Path, folds_csv: Path, extra_folds_csv: Path, mvd_folds_csv: Path | None):
        u = pd.read_parquet(units, columns=["unit_id", "func", "language", "target", "cvefixes_only_eval"])
        main = pd.read_csv(folds_csv)
        allv2 = pd.read_csv(extra_folds_csv)
        assert set(main["unit_id"]) <= set(allv2["unit_id"]), "tập chính phải là tập con của v2"
        self.cvf = S.attach_folds(u[u["unit_id"].isin(main["unit_id"])][["unit_id", "func", "language", "target"]]
                                  .reset_index(drop=True), main)
        ext = allv2[~allv2["unit_id"].isin(main["unit_id"])]
        self.extra = S.attach_folds(u[u["unit_id"].isin(ext["unit_id"])][["unit_id", "func", "language", "target"]]
                                    .reset_index(drop=True), ext)
        ev = set(u.loc[u["cvefixes_only_eval"].astype(bool), "unit_id"])
        self.cvf_eval = self.cvf[self.cvf["unit_id"].isin(ev)].reset_index(drop=True)
        self.mvd_folds_csv = mvd_folds_csv
        self._mvd = None
        print(f"CVEfixes: tập chính {len(self.cvf)} hàm, thêm {len(self.extra)} hàm v2 ngoài tập chính, "
              f"{len(self.cvf_eval)} hàm chỉ có ở CVEfixes", flush=True)

    @property
    def mvd(self) -> pd.DataFrame:
        if self._mvd is None:
            folds = pd.read_csv(self.mvd_folds_csv)
            clean = MVDDatasetLoader().clean_exact_hash(copy=True)
            clean = clean[clean["unit_id"].isin(folds["unit_id"])].reset_index(drop=True)
            self._mvd = S.attach_folds(clean, folds)
            print(f"MVD v2: {len(self._mvd)} hàm", flush=True)
        return self._mvd

    def parts(self, mode: str, key: str, k: int) -> dict:
        if mode in ("mono", "multi", "lolo"):
            tr, va, te, removed = S.get_split(self.cvf, mode, key, k)
            extra = None
            if mode == "mono":
                extra = self.extra[(self.extra["fold"] == k) & (self.extra["language"] == key)].reset_index(drop=True)
            elif mode == "multi":
                extra = self.extra[self.extra["fold"] == k].reset_index(drop=True)
            if extra is not None and len(extra):
                # hàm thêm thuộc cụm của fold k -> không được trùng cụm với train/val của job này
                assert not (set(extra["group"]) & (set(tr["group"]) | set(va["group"]))), "RÒ RỈ cụm ở tập thêm"
            return dict(train=tr, val=va, test=te, extra=extra, removed=removed)
        if mode in ("xmono", "xmulti"):
            m = self.mvd if mode == "xmulti" else self.mvd[self.mvd["language"] == key]
            te = self.cvf_eval if mode == "xmulti" else self.cvf_eval[self.cvf_eval["language"] == key]
            va = m[m["inner_val_fold0"]].reset_index(drop=True)
            tr = m[~m["inner_val_fold0"]].reset_index(drop=True)
            assert not (set(tr["group"]) & set(va["group"])), "RÒ RỈ cụm giữa train và validation MVD"
            assert not (set(te["unit_id"]) & (set(tr["unit_id"]) | set(va["unit_id"]))), "hàm kiểm tra trùng hàm MVD"
            return dict(train=tr, val=va, test=te.reset_index(drop=True), extra=None, removed=0)
        raise ValueError(mode)


def all_job_names(data: Data):
    jobs = [f"{m}:{key}:{k}" for m, key, k in S.build_jobs(data.cvf)]
    jobs += ["xmulti:ALL:-1"] + [f"xmono:{lang}:-1" for lang in S.LANGUAGES]
    return jobs


A_TRAIN, B_INFER = 0.100, 0.0403   # giây/mẫu trên T4 với MVD (5 epoch, max_length 768)
LEN_CVF, LEN_EXTRA = 1.2, 1.6      # hệ số độ dài: hàm CVEfixes dài hơn MVD (ước lượng thận trọng từ phân bố token)


def estimate(e, mode, n_train, n_val, n_test, n_extra):
    if mode in ("xmono", "xmulti"):
        return e * n_train * A_TRAIN + e * n_val * B_INFER + n_test * B_INFER * LEN_CVF
    return (e * n_train * A_TRAIN + e * (n_val + n_test) * B_INFER) * LEN_CVF + n_extra * B_INFER * LEN_EXTRA


def smoke(data: Data, caches: dict, cfg, device, out: Path, report_dir: Path) -> None:
    """Chạy thử cả hai loại job trên mẫu rất nhỏ (2 epoch). Không dùng test thật của job CV; kết quả bị bỏ."""
    p = data.parts("mono", "Python", 0)
    va = p["val"]
    mini = dict(train=p["train"].head(48), val=va.head(24), test=va.iloc[24:48].reset_index(drop=True),
                extra=p["extra"].head(16), removed=0)
    r1 = run_job("smoke", "Python", 0, 0, mini, dict(train=caches["cvf"], test=caches["cvf"]), cfg, device, out, 2, True)
    jd = job_dir(out, "smoke", "Python", 0, 0, 2)
    pred = pd.read_csv(jd / "predictions.csv"); xp = pd.read_csv(jd / "extra_predictions.csv")
    tby = pd.read_csv(jd / "test_by_epoch.csv").set_index("unit_id")
    assert len(pred) == 24 and len(xp) == 16 and pred["prob_vulnerable"].between(0, 1).all()
    sel = f"p_e{r1['selected_epoch']}"
    assert np.allclose(pred.set_index("unit_id")["prob_vulnerable"].reindex(tby.index), tby[sel])
    p = data.parts("xmono", "Python", -1)
    mini = dict(train=p["train"].head(48), val=p["val"].head(24), test=p["test"].head(24), extra=None, removed=0)
    r2 = run_job("smokex", "Python", -1, 0, mini, dict(train=caches["mvd"], test=caches["cvf"]), cfg, device, out, 2, False)
    jd = job_dir(out, "smokex", "Python", -1, 0, 2)
    assert len(pd.read_csv(jd / "predictions.csv")) == 24 and not (jd / "test_by_epoch.csv").exists()
    info = {f"{n}_{k}": r[k] for n, r in (("cv", r1), ("cross", r2)) for k in ("n_train", "n_val", "n_test", "selected_epoch", "seconds")}
    report_dir.mkdir(parents=True, exist_ok=True)
    (report_dir / "smoke_report.json").write_text(json.dumps(dict(status="SMOKE OK", **info, env=env_info()), indent=2), encoding="utf-8")
    print("SMOKE OK:", info)


def parse_jobs(specs):
    out = []
    for s in specs:
        e, m, key, k = s.split(":")
        out.append((int(e), m, key, int(k)))
    return out


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--units", type=Path, required=True)
    ap.add_argument("--folds-csv", type=Path, required=True, help="groups_and_folds_cvefixes_v2_len768_paired.csv")
    ap.add_argument("--extra-folds-csv", type=Path, required=True, help="groups_and_folds_cvefixes_v2.csv")
    ap.add_argument("--mvd-folds-csv", type=Path, default=None, help="groups_and_folds_v2.csv của MVD (job x*)")
    ap.add_argument("--out", type=Path, default=Path("/kaggle/working/cvefixes_results"))
    ap.add_argument("--seeds", nargs="*", type=int, default=[17])
    ap.add_argument("--jobs", nargs="*", default=None, help='dạng "10:mono:C:0", "10:xmulti:ALL:-1"')
    ap.add_argument("--max-hours", type=float, default=8.3)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--report-dir", type=Path, default=Path("/kaggle/working/smoke_cvefixes"))
    ap.add_argument("--plan-only", action="store_true")
    args = ap.parse_args()

    cfg = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    data = Data(args.units, args.folds_csv, args.extra_folds_csv, args.mvd_folds_csv)
    valid = set(all_job_names(data))
    specs = parse_jobs(args.jobs or [f"10:{j}" for j in all_job_names(data)])
    todo = []
    for e, m, key, k in specs:
        assert f"{m}:{key}:{k}" in valid, f"job không tồn tại: {m}:{key}:{k}"
        if m.startswith("x") and args.mvd_folds_csv is None:
            raise SystemExit("job huấn luyện trên MVD cần --mvd-folds-csv")
        p = data.parts(m, key, k)
        nx = 0 if p["extra"] is None else len(p["extra"])
        todo.append((e, m, key, k, estimate(e, m, len(p["train"]), len(p["val"]), len(p["test"]), nx)))
    print(f"Số job yêu cầu: {len(todo)}  |  ước tính tổng {sum(t[4] for t in todo) / 3600:.2f} giờ GPU", flush=True)
    if args.plan_only:
        for e, m, key, k, est in todo:
            print(f"  {e:2d} epoch  {m}:{key}:{k}  ~{est / 3600:.2f} giờ")
        return 0

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Thiết bị:", device)
    pipe = UniXcoderInputPipeline()
    caches = dict(cvf=RecordCache(pipe), mvd=RecordCache(pipe))
    if args.smoke:
        import tempfile
        smoke(data, caches, cfg, device, Path(tempfile.mkdtemp()), args.report_dir)
        return 0

    start, act, pred_sum, left = time.time(), 0.0, 0.0, []
    for seed in args.seeds:
        for e, m, key, k, est in todo:
            d = job_dir(args.out, m, key, k, seed, e)
            if job_done(d):
                print(f"[BỎ QUA] đã xong: {d}")
                continue
            ratio = (act / pred_sum) if pred_sum > 0 else 1.0
            need = est * ratio * 1.1 / 3600
            elapsed = (time.time() - start) / 3600
            if elapsed + need > args.max_hours:
                print(f"[ĐỂ LẦN SAU] {e} epoch {m}/{key}/fold{k}: cần ~{need:.2f} giờ, còn {args.max_hours - elapsed:.2f} giờ")
                left.append(f"{e}:{m}:{key}:{k}")
                continue
            print(f"[BẮT ĐẦU] {e} epoch {m}/{key}/fold{k} (ước tính {need:.2f} giờ, đã chạy {elapsed:.2f} giờ)", flush=True)
            cross = m.startswith("x")
            c = dict(train=caches["mvd"] if cross else caches["cvf"], test=caches["cvf"])
            res = run_job(m, key, k, seed, data.parts(m, key, k), c, cfg, device, args.out, e, per_epoch_test=not cross)
            act += res["seconds"]
            pred_sum += est
    if left:
        print(f"CÒN {len(left)} job cho lần chạy sau: {left}")
    else:
        print("HOÀN TẤT các job được yêu cầu.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


## 6. (Chỉ dùng khi phiên bị ngắt giữa chừng) nạp kết quả dở dang
Bình thường cell này không làm gì. Nếu phiên bị ngắt: Add Input là output của bản commit bị ngắt rồi chạy lại; job đã xong được bỏ qua.

In [ ]:
OUT = Path('/kaggle/working/cvefixes_results'); OUT.mkdir(exist_ok=True)
n = 0
for p in [Path(x) for x in glob.glob('/kaggle/input/**/cvefixes_results', recursive=True)]:
    for f in p.rglob('*'):
        if f.is_file():
            dst = OUT / f.relative_to(p)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst); n += 1
for z in glob.glob('/kaggle/input/**/cvf_s43_nb2.zip', recursive=True):   # trường hợp input là file zip
    with zipfile.ZipFile(z) as zz:
        for name in zz.namelist():
            if name.startswith('cvefixes_results/') and not name.endswith('/'):
                dst = Path('/kaggle/working') / name
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True); dst.write_bytes(zz.read(name)); n += 1
runs_file = OUT / 'runs.txt'
RUN = (len(runs_file.read_text().split()) if runs_file.exists() else 0) + 1
print('Đã nạp', n, 'file dở dang' if n else 'file (bình thường, chạy mới)')

## 7. Chạy thử nhanh (khoảng 5 phút)
Chạy thử cả job CV (có dự đoán tập thêm) và job huấn luyện trên MVD, 2 epoch trên vài chục mẫu. Phải in `SMOKE OK`. Kết quả bị bỏ.

In [ ]:
import subprocess
DATA_ARGS = ['--units', UNITS, '--folds-csv', CVF_MAIN, '--extra-folds-csv', CVF_V2, '--mvd-folds-csv', FOLDS]
r = subprocess.run(['python', 'src/training/run_cvefixes.py', *DATA_ARGS, '--smoke'],
                   capture_output=True, text=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), cwd=str(REPO))
print(r.stdout[-3000:]); print(r.stderr[-3000:])
assert 'SMOKE OK' in r.stdout, 'Chạy thử thất bại: xem lỗi ở trên, chụp gửi lại'

## 8. Huấn luyện trên 2 GPU
Mỗi GPU chạy một danh sách job (ước tính 7.4 và 7.4 giờ). Cứ 5 phút in một dòng tiến độ.

In [ ]:
import subprocess, os, time
SESSION_HOURS = 8.3      # giới hạn an toàn cho phiên 9 giờ; tính từ lúc notebook bắt đầu
LANES = [["10:lolo:C++:-1", "10:mono:JavaScript:4", "10:mono:C++:3"], ["10:lolo:Java:-1", "10:mono:Java:2", "10:mono:C++:4", "10:mono:C#:2"]]
n_gpu = int(subprocess.check_output('nvidia-smi -L | wc -l', shell=True).decode().strip())
assert n_gpu == 2, f'Cần Accelerator = GPU T4 x2 (đang thấy {n_gpu} GPU). Settings → Accelerator.'
def remaining_hours():
    return max(0.1, SESSION_HOURS - (time.time() - NB_START) / 3600)
procs = []
for i, jobs in enumerate(LANES):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), TOKENIZERS_PARALLELISM='false')
    cmd = ['python', 'src/training/run_cvefixes.py', *DATA_ARGS, '--jobs', *jobs, '--seeds', '43',
           '--out', str(OUT), '--max-hours', f'{remaining_hours():.2f}']
    log = open(f'/kaggle/working/lane{i}_run{RUN}.log', 'w')
    procs.append(subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, cwd=str(REPO)))
    print(f'lane {i}: GPU {i}, {len(jobs)} job'); time.sleep(45)
def tail(i):
    try:
        t = open(f'/kaggle/working/lane{i}_run{RUN}.log', errors='ignore').read().split('\n')
        t = [x.split('\r')[-1] for x in t if x.strip()]
        return t[-1][:180] if t else ''
    except Exception as e:
        return str(e)
try:
    while any(p.poll() is None for p in procs):
        time.sleep(300)
        print(f'[{(time.time() - NB_START) / 3600:.2f} giờ]')
        for i in range(len(procs)): print(f'  lane{i}:', tail(i))
        print('  GPU:', subprocess.run('nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip().replace('\n', ' | '))
finally:
    for p in procs:
        if p.poll() is None: p.terminate()
print('Mã thoát:', [p.returncode for p in procs])
for i in range(len(procs)):
    print(f'--- cuối lane{i}_run{RUN}.log ---'); print('\n'.join([x for x in open(f'/kaggle/working/lane{i}_run{RUN}.log', errors='ignore').read().split('\n') if x.strip()][-5:]))

## 9. Kiểm tra và đóng gói (zip gồm cả kết quả các lần trước nếu có)

In [ ]:
import json
with open(OUT / 'runs.txt', 'a') as fh: fh.write(f'run{RUN}\n')
for i in range(2):
    src = Path(f'/kaggle/working/lane{i}_run{RUN}.log')
    if src.exists(): shutil.copy2(src, OUT / f'nb2_{src.name}')
ck = lambda k: k.replace('#', 'sharp').replace('+', 'plus').lower()
done, left = [], []
for j in [j for l in LANES for j in l]:
    e, m, k, f = j.split(':')
    d = OUT / f'e{e}' / m / ck(k) / (f'fold{f}' if int(f) >= 0 else 'all') / 'seed_43'
    need = ['predictions.csv', 'log.json', 'val_by_epoch.csv']
    if not m.startswith('x'): need.append('test_by_epoch.csv')
    if m in ('mono', 'multi'): need.append('extra_predictions.csv')
    ok = all((d / x).exists() for x in need)
    (done if ok else left).append(j)
    if ok:
        lg = json.load(open(d / 'log.json')); assert lg['max_epochs'] == int(e) and len(lg['epochs']) == int(e), d
        print(f"{m:6s} {k:10s} fold={f:>2s} epoch={lg['selected_epoch']:2d} {lg['seconds'] / 3600:.2f} giờ")
print(f'Đã xong {len(done)}/{len(done) + len(left)} job của notebook seed 43 số 2')
if left:
    print('CÒN LẠI:', left)
    print('→ Gửi file zip này kèm ảnh chụp, mình sẽ hướng dẫn chạy nốt.')
else:
    print('NOTEBOOK seed 43 số 2 ĐÃ XONG TOÀN BỘ. Gửi file cvf_s43_nb2.zip.')
name = 'cvf_s43_nb2'
shutil.make_archive(f'/kaggle/working/{name}', 'zip', '/kaggle/working', 'cvefixes_results')
from IPython.display import FileLink, display
display(FileLink(f'/kaggle/working/{name}.zip'))